<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03_feature_engineering**

# **0. Configuración del Entorno**


## 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 0.2. Instalación e importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

## 0.4. Definición de rutas



In [5]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [6]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = DRIVE_DIR / Path(os.environ.get("IN_PARQUET", "data/03_targets/mnq_intraday_t2.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = DRIVE_DIR / Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

## 0.5. Códigos auxiliares para carga de datos y visualización


In [7]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [8]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

In [17]:
mnq_intraday = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (1024062, 15)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id', 'close_fwd_30', 'delta_30', 'close_fwd_60', 'delta_60', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2026-04-17 16:00:00-04:00
First/Last day: 2020-01-02  ->  2026-04-17
Total days (trading): 1482
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2026-04-17 20:00:00+00:00


## 0.6. Auxiliares

In [18]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [11]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

# **1. Indicadores Técnicos**

In [14]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

Librería instalada: technical-analysis


In [15]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

## **1.1. Indicadores técnicos individuales**

#### 1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [19]:
def calcular_rsi(df=mnq_intraday, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

#### 2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [20]:
def calcular_momentum(df=mnq_intraday, target='close' ):
  momentum_columns = ['mom_10', 'mom_5','mom_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['mom_10'] = grupo[target].pct_change(10)
        grupo['mom_5'] = grupo[target].pct_change(5)
        grupo['mom_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

#### 3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [21]:
def calcular_volumen_ratio(df=mnq_intraday, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

#### 4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [22]:
def calcular_macd(df=mnq_intraday, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

#### 5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [23]:
def calcular_ema(df=mnq_intraday, target='close'):
    ema_columns = ['ema_15', 'ema_20', 'ema_30',  'ema_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['ema_15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['ema_20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['ema_30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['ema_60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

#### 6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [24]:
def calcular_stochastic(df=mnq_intraday, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


#### 7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [25]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [26]:
def calcular_bollinger_resume(df=mnq_intraday, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

#### 8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [27]:
def calcular_atr(df=mnq_intraday, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

####  9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [28]:
def calcular_roc(df=mnq_intraday, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

## **1.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [29]:
import os
import glob
import numpy as np
import pandas as pd


def load_or_build_mnq_intraday_with_indicators(
    mnq_intraday: pd.DataFrame,
    *,
    processed_dir: str = "/content/drive/MyDrive/neural_profit/data/03_targets",
    parquet_name: str = "mnq_intraday_tech_indicators.parquet",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    regime_col: str = "regime_id",
    valid_regime_ids: tuple[int, ...] = (0, 1, 2, 3, 4),
    dropna_after_indicators: bool = True,
    print_report: bool = True,
):
    """
    Carga un parquet existente con indicadores o los calcula desde mnq_intraday.

    Además:
    - elimina NaNs al final del cálculo
    - reconstruye indicator_columns
    - valida orden cronológico
    - valida NaNs remanentes en indicadores
    - valida consistencia básica por día para indicadores rolling
    - imprime un reporte final del dataset
    """

    processed_dir = os.path.abspath(processed_dir)
    parquet_path = os.path.join(processed_dir, parquet_name)
    os.makedirs(processed_dir, exist_ok=True)

    # ============================================================
    # 1) Cargar o calcular
    # ============================================================
    loaded_from = None

    if os.path.exists(parquet_path):
        df = pd.read_parquet(parquet_path)
        loaded_from = parquet_path
        if print_report:
            print(f"[OK] Cargado: {parquet_path}")

    else:
        candidates = sorted(glob.glob(os.path.join(processed_dir, "*with_indicators*.parquet")))
        if candidates:
            df = pd.read_parquet(candidates[-1])
            loaded_from = candidates[-1]
            if print_report:
                print(f"[OK] Cargado (fallback): {candidates[-1]}")
        else:
            df = mnq_intraday.copy()
            loaded_from = "computed"

            if print_report:
                print("[OK] Calculando indicadores técnicos...")

            df, rsi_columns = calcular_rsi(df)
            df, momentum_columns = calcular_momentum(df)
            df, volume_ratio_columns = calcular_volumen_ratio(df)
            df, macd_columns = calcular_macd(df)
            df, ema_columns = calcular_ema(df)
            df, stoch_columns = calcular_stochastic(df)
            df, bollinger_columns = calcular_bollinger(df)
            df, atr_columns = calcular_atr(df)
            df, roc_columns = calcular_roc(df)

            indicator_columns = (
                rsi_columns
                + momentum_columns
                + volume_ratio_columns
                + macd_columns
                + ema_columns
                + stoch_columns
                + bollinger_columns
                + atr_columns
                + roc_columns
            )

            indicator_columns = list(dict.fromkeys(indicator_columns))

            # ============================================================
            # 2) Eliminar NaNs SOLO al final
            # ============================================================
            if dropna_after_indicators:
                n_before_dropna = len(df)
                df = df.dropna(subset=indicator_columns).copy()
                n_after_dropna = len(df)
            else:
                n_before_dropna = len(df)
                n_after_dropna = len(df)

            df.to_parquet(parquet_path, index=True)
            if print_report:
                print(f"[OK] Calculado y guardado: {parquet_path}")

    # ============================================================
    # 3) Normalización temporal
    # ============================================================
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])

    df = df.sort_index().copy()

    # ============================================================
    # 4) Reconstruir indicator_columns desde el dataset final
    # ============================================================
    columns_to_remove = {
        date_col,
        minute_col,
        regime_col,
        "open", "high", "low", "close", "volume",
    }

    indicator_columns = [col for col in df.columns if col not in columns_to_remove]

    # ============================================================
    # 5) Validaciones
    # ============================================================
    # 5.1 Orden cronológico
    is_sorted = df.index.is_monotonic_increasing

    # 5.2 NaNs en indicadores
    nan_counts_indicators = df[indicator_columns].isna().sum().sort_values(ascending=False)
    indicators_with_nan = nan_counts_indicators[nan_counts_indicators > 0]

    # 5.3 Filas por regime_id
    if regime_col in df.columns:
        regime_summary = (
            df[regime_col]
            .value_counts(dropna=False)
            .sort_index()
            .rename_axis(regime_col)
            .reset_index(name="n_rows")
        )
    else:
        regime_summary = pd.DataFrame(columns=[regime_col, "n_rows"])

    # 5.4 Validación básica: regime_id válido por fila
    if regime_col in df.columns:
        invalid_regime_rows = int(~df[regime_col].isin(valid_regime_ids).sum())  # no usar
        invalid_regime_rows = int((~df[regime_col].isin(valid_regime_ids)).sum())
    else:
        invalid_regime_rows = np.nan

    # 5.5 Validación básica: no cruce de días en indicadores
    day_minute_summary = (
        df.groupby(date_col)[minute_col]
        .agg(first_minute="min", last_minute="max", n_rows="count")
        .reset_index()
        if date_col in df.columns and minute_col in df.columns
        else pd.DataFrame()
    )

    suspicious_cross_day_days = pd.DataFrame()
    if not day_minute_summary.empty:
        suspicious_cross_day_days = day_minute_summary[
            day_minute_summary["first_minute"] < 60
        ].copy()

    # 5.6 Duplicados temporales
    duplicated_timestamps = int(df.index.duplicated().sum())

    # ============================================================
    # 6) Reporte
    # ============================================================
    report = {
        "loaded_from": loaded_from,
        "n_rows_final": int(len(df)),
        "n_cols_final": int(df.shape[1]),
        "n_indicator_columns": int(len(indicator_columns)),
        "is_sorted_chronologically": bool(is_sorted),
        "duplicated_timestamps": duplicated_timestamps,
        "n_indicators_with_nan": int(len(indicators_with_nan)),
        "invalid_regime_rows": invalid_regime_rows,
        "n_suspicious_cross_day_days": int(len(suspicious_cross_day_days)),
    }

    if loaded_from == "computed":
        report["n_rows_before_dropna"] = int(n_before_dropna)
        report["n_rows_after_dropna"] = int(n_after_dropna)
        report["rows_removed_by_dropna"] = int(n_before_dropna - n_after_dropna)

    if print_report:
        print("=" * 100)
        print("DATASET FINAL CON INDICADORES TÉCNICOS")
        print("=" * 100)
        print(f"Origen: {report['loaded_from']}")
        print(f"Filas finales: {report['n_rows_final']}")
        print(f"Columnas finales: {report['n_cols_final']}")
        print(f"Cantidad de indicadores técnicos: {report['n_indicator_columns']}")
        if "n_rows_before_dropna" in report:
            print(f"Filas antes de dropna final: {report['n_rows_before_dropna']}")
            print(f"Filas después de dropna final: {report['n_rows_after_dropna']}")
            print(f"Filas eliminadas por NaNs: {report['rows_removed_by_dropna']}")
        print(f"Orden cronológico correcto: {report['is_sorted_chronologically']}")
        print(f"Timestamps duplicados: {report['duplicated_timestamps']}")
        print(f"Indicadores con NaNs remanentes: {report['n_indicators_with_nan']}")
        print(f"Filas con regime_id inválido: {report['invalid_regime_rows']}")
        print(f"Días sospechosos de cruce entre días: {report['n_suspicious_cross_day_days']}")
        print()

        print("-" * 100)
        print("FILAS POR REGIME_ID")
        print("-" * 100)
        print(regime_summary.to_string(index=False))
        print()

        print("-" * 100)
        print("LISTADO TOTAL DE INDICADORES TÉCNICOS")
        print("-" * 100)
        print(indicator_columns)
        print()

        if len(indicators_with_nan) == 0:
            print("[OK] No quedan NaNs en los indicadores técnicos.")
        else:
            print("[WARN] Quedan NaNs en algunos indicadores:")
            print(indicators_with_nan.to_string())
            print()

        if is_sorted:
            print("[OK] El dataset final está en orden cronológico.")
        else:
            print("[WARN] El dataset final NO está en orden cronológico.")

        if duplicated_timestamps == 0:
            print("[OK] No hay timestamps duplicados.")
        else:
            print(f"[WARN] Hay {duplicated_timestamps} timestamps duplicados.")

        if invalid_regime_rows == 0:
            print("[OK] Todos los valores de regime_id son válidos.")
        else:
            print(f"[WARN] Hay {invalid_regime_rows} filas con regime_id inválido.")

        if len(suspicious_cross_day_days) == 0:
            print("[OK] No se detectaron señales evidentes de cruce entre días en indicadores.")
        else:
            print("[WARN] Hay días potencialmente sospechosos respecto al arranque de indicadores:")
            print(suspicious_cross_day_days.head(20).to_string(index=False))

    return df, indicator_columns, {
        "report": report,
        "regime_summary": regime_summary,
        "indicator_columns": indicator_columns,
        "nan_counts_indicators": indicators_with_nan,
        "suspicious_cross_day_days": suspicious_cross_day_days,
        "day_minute_summary": day_minute_summary,
    }

In [30]:
mnq_intraday_ti, indicator_columns, indicators_report = load_or_build_mnq_intraday_with_indicators(
    mnq_intraday,
    print_report=True,
)

[OK] Calculando indicadores técnicos...
[OK] Calculado y guardado: /content/drive/MyDrive/neural_profit/data/03_targets/mnq_intraday_tech_indicators.parquet
DATASET FINAL CON INDICADORES TÉCNICOS
Origen: computed
Filas finales: 892164
Columnas finales: 57
Cantidad de indicadores técnicos: 49
Filas antes de dropna final: 1024062
Filas después de dropna final: 892164
Filas eliminadas por NaNs: 131898
Orden cronológico correcto: True
Timestamps duplicados: 0
Indicadores con NaNs remanentes: 7
Filas con regime_id inválido: 0
Días sospechosos de cruce entre días: 0

----------------------------------------------------------------------------------------------------
FILAS POR REGIME_ID
----------------------------------------------------------------------------------------------------
 regime_id  n_rows
         0  225264
         1   88920
         2   88920
         3  444600
         4   44460

-----------------------------------------------------------------------------------------------

# **2. Análisis del dataset de entrada y targets**

In [33]:
mnq_intraday_t2_ti = mnq_intraday_ti.copy()

In [39]:
import numpy as np
import pandas as pd


def analyze_input_dataset_t2_ti(
    df: pd.DataFrame,
    *,
    datetime_index_required: bool = True,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    regime_col: str = "regime_id",
    ohlcv_cols: tuple[str, ...] = ("open", "high", "low", "close", "volume"),
    ti_cols: tuple[str, ...] = (
        "rsi_14", "rsi_7", "rsi_5", "rsi_3",
        "mom_10", "mom_5", "mom_3",
        "volume_ratio_15", "volume_ratio_20", "volume_ratio_30",
        "volume_ratio_60", "volume_ratio_90",
        "macd",
        "ema_15", "ema_20", "ema_30", "ema_60",
        "stoch_k_14", "stoch_k_20", "stoch_k_30",
        "bb_15_15", "bb_20_15", "bb_30_15", "bb_60_15",
        "bb_15_20", "bb_20_20", "bb_30_20", "bb_60_20",
        "bb_15_25", "bb_20_25", "bb_30_25", "bb_60_25",
        "atr_norm_5", "atr_norm_10", "atr_norm_14", "atr_norm_20", "atr_norm_30",
        "roc_5", "roc_10", "roc_20", "roc_30", "roc_60",
    ),
    target_cols: tuple[str, ...] = (
        't2_p40_h30',
        't2_p40_h60',
        't2_p50_h30',
    ),
    print_report: bool = True,
) -> dict:
    """
    Analiza el dataset mnq_intraday_t2_ti antes de modelado.

    Verifica:
    - presencia de columnas esperadas
    - orden temporal correcto
    - consistencia básica OHLCV
    - consistencia de regime_id
    - ausencia básica de leakage temporal en targets T2
    - distribución multiclase de los targets
    - NaNs en indicadores técnicos
    - duplicados temporales

    Retorna un diccionario con:
    - report
    - target_summary
    - regime_summary
    - ti_nan_summary
    - invalid_ohlc_rows
    - invalid_regime_rows
    - leakage_rows
    """

    data = df.copy()

    # ============================================================
    # 0. Validación de columnas
    # ============================================================
    required_cols = set(ohlcv_cols) | set(ti_cols) | set(target_cols) | {date_col, minute_col, regime_col}
    missing_cols = sorted([c for c in required_cols if c not in data.columns])

    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    # ============================================================
    # 1. Validación / normalización temporal
    # ============================================================
    if datetime_index_required and not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener DatetimeIndex.")

    data[date_col] = pd.to_datetime(data[date_col])

    if isinstance(data.index, pd.DatetimeIndex):
        data = data.sort_index().copy()
        is_sorted = data.index.is_monotonic_increasing
    else:
        data = data.sort_values([date_col, minute_col]).copy()
        is_sorted = (
            data[[date_col, minute_col]]
            .reset_index(drop=True)
            .equals(
                data[[date_col, minute_col]]
                .sort_values([date_col, minute_col])
                .reset_index(drop=True)
            )
        )

    # ============================================================
    # 2. Consistencia OHLCV
    # ============================================================
    invalid_ohlc_mask = (
        (data["high"] < data["low"]) |
        (data["open"] > data["high"]) |
        (data["open"] < data["low"]) |
        (data["close"] > data["high"]) |
        (data["close"] < data["low"]) |
        (data["volume"] < 0)
    )
    invalid_ohlc_rows = data.loc[invalid_ohlc_mask].copy()

    # ============================================================
    # 3. Consistencia de regime_id
    # ============================================================
    invalid_regime_mask = data[regime_col].isna()
    invalid_regime_rows = data.loc[invalid_regime_mask].copy()

    regime_summary = (
        data[regime_col]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis(regime_col)
        .reset_index(name="n_rows")
    )

    # ============================================================
    # 4. Leakage temporal básico
    #    Si no hay futuro suficiente dentro del día, el target debería ser NaN
    # ============================================================
    if not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("Para validar leakage temporal, el DataFrame debe tener DatetimeIndex.")

    leakage_checks = []

    last_ts_by_day = data.groupby(date_col).apply(lambda g: g.index.max())
    last_ts_by_day.name = "last_ts_day"

    data = data.merge(
        last_ts_by_day,
        left_on=date_col,
        right_index=True,
        how="left"
    )

    for target_col in target_cols:
        horizon_str = target_col.split("_")[-1]

        if horizon_str.startswith("h"):
            horizon = int(horizon_str[1:])
        else:
            horizon = int(horizon_str)

        expected_future_ts = data.index + pd.Timedelta(minutes=horizon)

        invalid_future_mask = expected_future_ts > data["last_ts_day"]
        suspect_non_nan_target_mask = invalid_future_mask & data[target_col].notna()

        leakage_checks.append(
            data.loc[suspect_non_nan_target_mask, [date_col, minute_col, target_col]].assign(
                target_col_name=target_col
            )
        )

    leakage_rows = pd.concat(leakage_checks, axis=0) if leakage_checks else pd.DataFrame()

    # ============================================================
    # 5. Resumen de targets T2 (multiclase)
    # ============================================================
    target_summary_rows = []

    for target_col in target_cols:
        s = data[target_col]

        valid = s.dropna()
        n_total = len(s)
        n_notna = valid.shape[0]
        n_nan = s.isna().sum()

        n_neg1 = (valid == -1).sum()
        n_0 = (valid == 0).sum()
        n_pos1 = (valid == 1).sum()

        target_summary_rows.append({
            "target_col": target_col,
            "n_total": int(n_total),
            "n_notna": int(n_notna),
            "n_nan": int(n_nan),
            "pct_nan": float(n_nan / n_total),
            "n_-1": int(n_neg1),
            "n_0": int(n_0),
            "n_1": int(n_pos1),
            "pct_-1": float(n_neg1 / n_notna) if n_notna > 0 else np.nan,
            "pct_0": float(n_0 / n_notna) if n_notna > 0 else np.nan,
            "pct_1": float(n_pos1 / n_notna) if n_notna > 0 else np.nan,
            "majority_class": valid.value_counts().idxmax() if n_notna > 0 else np.nan,
        })

    target_summary = pd.DataFrame(target_summary_rows)

    # ============================================================
    # 6. NaNs en indicadores técnicos
    # ============================================================
    ti_nan_summary = pd.DataFrame({
        "feature": list(ti_cols),
        "n_nan": [int(data[c].isna().sum()) for c in ti_cols],
        "pct_nan": [float(data[c].isna().mean()) for c in ti_cols],
    }).sort_values(["pct_nan", "feature"], ascending=[False, True])

    # ============================================================
    # 7. Duplicados temporales
    # ============================================================
    if isinstance(data.index, pd.DatetimeIndex):
        dup_time_count = int(data.index.duplicated().sum())
    else:
        dup_time_count = int(data.duplicated(subset=[date_col, minute_col]).sum())

    # ============================================================
    # 8. Resumen general
    # ============================================================
    report = {
        "n_rows": int(len(data)),
        "n_cols": int(data.shape[1]),
        "datetime_index_type": str(type(data.index)),
        "is_sorted_temporally": bool(is_sorted),
        "n_unique_days": int(data[date_col].nunique()),
        "datetime_duplicates": dup_time_count,
        "n_invalid_ohlc_rows": int(len(invalid_ohlc_rows)),
        "n_invalid_regime_rows": int(len(invalid_regime_rows)),
        "n_suspect_leakage_rows": int(len(leakage_rows)),
        "n_ti_features": int(len(ti_cols)),
    }

    # ============================================================
    # 9. Print reporte
    # ============================================================
    if print_report:
        print("=" * 100)
        print("ANÁLISIS DEL DATASET | mnq_intraday_t2_ti")
        print("=" * 100)
        print(f"Filas: {report['n_rows']:,}")
        print(f"Columnas: {report['n_cols']}")
        print(f"Días únicos: {report['n_unique_days']}")
        print(f"Orden temporal correcto: {report['is_sorted_temporally']}")
        print(f"Timestamps duplicados: {report['datetime_duplicates']}")
        print(f"Filas con OHLCV inconsistente: {report['n_invalid_ohlc_rows']}")
        print(f"Filas con regime_id inválido: {report['n_invalid_regime_rows']}")
        print(f"Filas sospechosas de leakage temporal: {report['n_suspect_leakage_rows']}")
        print(f"Número de indicadores técnicos: {report['n_ti_features']}")
        print()

        print("-" * 100)
        print("MUESTRAS POR RÉGIMEN")
        print("-" * 100)
        print(regime_summary.to_string(index=False))
        print()

        print("-" * 100)
        print("RESUMEN DE TARGETS T2")
        print("-" * 100)
        print(target_summary.to_string(index=False))
        print()

        print("-" * 100)
        print("TOP 15 FEATURES CON MÁS NaNs")
        print("-" * 100)
        print(ti_nan_summary.head(15).to_string(index=False))
        print()

        if len(invalid_ohlc_rows) > 0:
            print("-" * 100)
            print("MUESTRA DE FILAS CON OHLCV INVÁLIDO")
            print("-" * 100)
            print(invalid_ohlc_rows.head(10).to_string())

        if len(invalid_regime_rows) > 0:
            print("-" * 100)
            print("MUESTRA DE FILAS CON REGIME_ID INVÁLIDO")
            print("-" * 100)
            print(invalid_regime_rows.head(10).to_string())

        if len(leakage_rows) > 0:
            print("-" * 100)
            print("MUESTRA DE FILAS SOSPECHOSAS DE LEAKAGE")
            print("-" * 100)
            print(leakage_rows.head(10).to_string(index=False))

    # limpieza
    if "last_ts_day" in data.columns:
        data = data.drop(columns=["last_ts_day"])

    return {
        "report": report,
        "target_summary": target_summary,
        "regime_summary": regime_summary,
        "ti_nan_summary": ti_nan_summary,
        "invalid_ohlc_rows": invalid_ohlc_rows,
        "invalid_regime_rows": invalid_regime_rows,
        "leakage_rows": leakage_rows,
    }

In [40]:
time_cols = ["date", "minute_of_day"]
regimen_cols = ["regime_id"]

ohlcv_cols = ["open", "high", "low", "close", "volume"]

ti_cols = [
    "rsi_14", "rsi_7", "rsi_5", "rsi_3", "mom_10", "mom_5", "mom_3",
    "volume_ratio_15", "volume_ratio_20", "volume_ratio_30",
    "volume_ratio_60", "volume_ratio_90", "macd", "ema_15", "ema_20",
    "ema_30", "ema_60", "stoch_k_14", "stoch_k_20", "stoch_k_30",
    "bb_15_15", "bb_20_15", "bb_30_15", "bb_60_15", "bb_15_20", "bb_20_20",
    "bb_30_20", "bb_60_20", "bb_15_25", "bb_20_25", "bb_30_25", "bb_60_25",
    "atr_norm_5", "atr_norm_10", "atr_norm_14", "atr_norm_20", "atr_norm_30",
    "roc_5", "roc_10", "roc_20", "roc_30", "roc_60",
]

targets_cols = [
        't2_p40_h30',
        't2_p40_h60',
        't2_p50_h30',
]

dataset_report = analyze_input_dataset_t2_ti(
    mnq_intraday_t2_ti,
    date_col="date",
    minute_col="minute_of_day",
    regime_col="regime_id",
    ohlcv_cols=tuple(ohlcv_cols),
    ti_cols=tuple(ti_cols),
    target_cols=tuple(targets_cols),
    print_report=True,
)

ANÁLISIS DEL DATASET | mnq_intraday_t2_ti
Filas: 892,164
Columnas: 58
Días únicos: 1482
Orden temporal correcto: True
Timestamps duplicados: 0
Filas con OHLCV inconsistente: 0
Filas con regime_id inválido: 0
Filas sospechosas de leakage temporal: 0
Número de indicadores técnicos: 42

----------------------------------------------------------------------------------------------------
MUESTRAS POR RÉGIMEN
----------------------------------------------------------------------------------------------------
 regime_id  n_rows
         0  225264
         1   88920
         2   88920
         3  444600
         4   44460

----------------------------------------------------------------------------------------------------
RESUMEN DE TARGETS T2
----------------------------------------------------------------------------------------------------
target_col  n_total  n_notna  n_nan  pct_nan  n_-1   n_0   n_1   pct_-1    pct_0    pct_1  majority_class
t2_p40_h30   892164   177840 714324 0.800664 51

# **3. Separación IS vs OOS**

En esta etapa se realiza la separación del dataset en dos subconjuntos temporales: **in-sample (IS)** y **out-of-sample (OOS)**.
Esta división se efectúa de manera estrictamente cronológica, respetando el orden temporal de los datos y evitando cualquier tipo de filtración de información futura.

El conjunto **in-sample (IS)** se utiliza para:

* el cálculo y selección de indicadores técnicos,
* el análisis del Information Coefficient (IC),
* la evaluación de dependencias y colinealidad entre features.

El conjunto **out-of-sample (OOS)** se reserva exclusivamente para:

* validar la estabilidad temporal de las relaciones observadas,
* comprobar que la capacidad predictiva de los indicadores no es producto del sobreajuste,
* verificar que las señales seleccionadas mantienen poder explicativo en datos no vistos.

Esta separación es un paso crítico para garantizar la validez estadística del proceso de feature engineering.
Los indicadores se seleccionan en base a su desempeño en el conjunto IS y se validan posteriormente en OOS.
Un indicador solo se considera robusto si mantiene un comportamiento consistente fuera de muestra, tanto en el signo como en la estabilidad de la magnitud del IC.

De este modo, la selección final de features se basa en criterios de **robustez temporal**, y no únicamente en el desempeño observado dentro del período de entrenamiento.

**Criterio propuesto (ajustable)**

- IS: 2020-01-02 a 2023-12-31
- OOS: 2024-01-01 a 2026-04-17

Este split es solo para selección y validación de features, no es el split final de modelado.

In [68]:
import numpy as np
import pandas as pd

def add_is_oos_split(
    df: pd.DataFrame,
    *,
    cut_date: str | pd.Timestamp = "2022-12-31",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    split_col: str = "split_fe",
    sort_before_split: bool = True,
    overwrite: bool = True,
    print_report: bool = True,
) -> pd.DataFrame:
    """
    Agrega una columna de split IS/OOS al dataset de entrada.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset de entrada, en este caso mnq_intraday_with_indicators.
    cut_date : str | pd.Timestamp
        Fecha de corte. <= cut_date -> IS, > cut_date -> OOS.
    date_col : str
        Nombre de la columna de fecha.
    minute_col : str
        Nombre de la columna de minuto intradía.
    split_col : str
        Nombre de la columna de salida para el split.
    sort_before_split : bool
        Si True, ordena cronológicamente antes de crear el split.
    overwrite : bool
        Si False y split_col ya existe, lanza error.
    print_report : bool
        Si True, imprime un resumen del split.

    Devuelve
    --------
    pd.DataFrame
        DataFrame con la columna split_fe agregada.
    """
    out = df.copy()

    if split_col in out.columns and not overwrite:
        raise ValueError(f"La columna '{split_col}' ya existe y overwrite=False.")

    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' en el DataFrame.")

    out[date_col] = pd.to_datetime(out[date_col])
    cut_date = pd.to_datetime(cut_date)

    # ============================================================
    # 1) Orden cronológico
    # ============================================================
    if sort_before_split:
        if isinstance(out.index, pd.DatetimeIndex):
            out = out.sort_index().copy()
        else:
            sort_cols = [date_col]
            if minute_col in out.columns:
                sort_cols.append(minute_col)
            out = out.sort_values(by=sort_cols).copy()

    # Validación
    if isinstance(out.index, pd.DatetimeIndex):
        is_sorted = out.index.is_monotonic_increasing
    else:
        if minute_col in out.columns:
            is_sorted = (
                out[[date_col, minute_col]]
                .reset_index(drop=True)
                .equals(
                    out[[date_col, minute_col]]
                    .sort_values([date_col, minute_col])
                    .reset_index(drop=True)
                )
            )
        else:
            is_sorted = out[date_col].is_monotonic_increasing

    if not is_sorted:
        raise ValueError("El dataset no está ordenado cronológicamente antes del split.")

    # ============================================================
    # 2) Crear split IS / OOS
    # ============================================================
    out[split_col] = np.where(
        out[date_col] <= cut_date,
        "IS",
        "OOS",
    )

    # ============================================================
    # 3) Reporte
    # ============================================================
    if print_report:
        split_summary = (
            out.groupby(split_col)
            .agg(
                n_rows=(split_col, "size"),
                start_date=(date_col, "min"),
                end_date=(date_col, "max"),
                n_days=(date_col, "nunique"),
            )
            .reset_index()
        )

        print("=" * 90)
        print("SEPARACIÓN IS vs OOS")
        print("=" * 90)
        print(f"Fecha de corte: {cut_date.date()}")
        print(f"Orden cronológico correcto: {is_sorted}")
        print()
        print(split_summary.to_string(index=False))

    return out


In [69]:
mnq_intraday_t2_ti = add_is_oos_split(
    mnq_intraday_t2_ti,
    cut_date="2023-12-31",
    print_report=True,
)

SEPARACIÓN IS vs OOS
Fecha de corte: 2023-12-31
Orden cronológico correcto: True

split_fe  n_rows start_date   end_date  n_days
      IS  569492 2020-01-02 2023-12-29     946
     OOS  322672 2024-01-02 2026-04-17     536


#**4. Evaluación mediante Information Coefficient (IC)**

## **4.1. Marco teórico**

### **4.1.1. Target T2 (dirección con umbral)**

En el dataset se utilizan como variables objetivo los targets discretos T2, definidos como:

- t2_dir_thr_h ∈ { -1, 0, 1 }

donde:

- 1  → movimiento positivo significativo  
- -1 → movimiento negativo significativo  
- 0  → movimiento no significativo (ruido)

Estos targets se construyen a partir de los deltas futuros (delta_h), aplicando un umbral definido mediante percentiles de la distribución de |delta_h|.

A diferencia de los targets continuos (delta_h), T2 no busca modelar la magnitud exacta del movimiento, sino identificar si el movimiento es lo suficientemente relevante como para ser considerado potencialmente operable.

De esta forma, T2 permite:

- filtrar el ruido de baja magnitud,
- concentrar el análisis en eventos significativos,
- mejorar la relación señal/ruido del problema.

---

Para evaluar la relación entre los indicadores técnicos y el target T2, se utiliza el coeficiente de correlación de Spearman.

Aunque T2 es una variable discreta, el uso de Spearman sigue siendo adecuado porque:

- no asume relaciones lineales,
- es robusto a outliers,
- captura relaciones monotónicas entre variables continuas (indicadores) y el target ordinal (-1 < 0 < 1).

---

Interpretación del IC (Information Coefficient):

- IC > 0  
  → el indicador tiende a tomar valores altos cuando ocurren movimientos positivos significativos (clase 1)

- IC < 0  
  → el indicador tiende a tomar valores altos cuando ocurren movimientos negativos significativos (clase -1)

- IC ≈ 0  
  → el indicador no muestra relación clara con el target

---

Es importante destacar que, en este contexto, el IC no mide la capacidad de predecir con precisión la clase, sino la existencia de una relación estadística entre el indicador y la ocurrencia de movimientos significativos.

Por lo tanto, valores de IC relativamente bajos (por ejemplo, en el rango 0.02–0.05) pueden ser relevantes en el contexto de datos financieros intradía.


### **4.1.2. Criterio metodológico — Evaluación por régimen e IS/OOS**


El IC se calcula de forma segmentada considerando dos dimensiones:

- régimen de mercado (según `regime_id`)
- partición temporal (`split_fe`: IS / OOS)

Este enfoque permite:

- identificar factores que mantienen señal en distintos contextos intradía,
- detectar dependencias específicas por régimen (ej: opening vs overnight),
- evaluar la estabilidad de la señal fuera de muestra (OOS),
- evitar sobreajuste a condiciones particulares del mercado.

De esta forma, cada indicador es evaluado no solo por su capacidad predictiva, sino por su **robustez estructural**.

### **4.1.3. Uso de los resultados**

Las tablas de IC obtenidas permiten:

- identificar indicadores con señal consistente en OOS,
- comparar el comportamiento de los indicadores entre regímenes,
- evaluar estabilidad entre IS y OOS (gap de generalización),
- detectar indicadores inestables o dependientes de condiciones específicas.

En particular, se priorizan indicadores que:

- mantienen signo consistente entre IS y OOS,
- presentan diferencias pequeñas entre ambos (bajo gap),
- muestran señal en múltiples regímenes.

### **4.1.4. Síntesis**


El Information Coefficient permite medir de forma directa la relación entre los indicadores técnicos y la ocurrencia de movimientos significativos del mercado.

Su evaluación conjunta por régimen de mercado y partición temporal (IS/OOS) constituye un criterio fundamental para la selección de factores:

- robustos,
- consistentes,
- y generalizables.

Este proceso permite reducir el conjunto de indicadores, eliminando redundancia y priorizando aquellos con mayor capacidad de aportar señal real al modelo.

## **4.2. Implementación de cálculo de IC**

### **4.2.1. Marcas temporales de régimen**

In [70]:
import pandas as pd


def build_regime_ranges_table(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    regime_col: str = "regime_id",
    regime_mapping: dict | None = None,
) -> pd.DataFrame:
    """
    Construye una tabla resumen por régimen (regime_id) con:

    - régimen (id o nombre)
    - minuto inicial
    - minuto final
    - hora inicial HH:MM
    - hora final HH:MM
    - cantidad de filas
    """

    if minute_col not in df.columns:
        raise KeyError(f"Falta la columna '{minute_col}' en el DataFrame.")

    if regime_col not in df.columns:
        raise KeyError(f"Falta la columna '{regime_col}' en el DataFrame.")

    rows = []

    # recorrer cada régimen único
    for regime_id in sorted(df[regime_col].dropna().unique()):
        m = df.loc[df[regime_col] == regime_id, minute_col].dropna()

        if m.empty:
            continue

        start_min = int(m.min())
        end_min = int(m.max())

        # nombre del régimen (opcional)
        regime_name = (
            regime_mapping.get(regime_id, regime_id)
            if regime_mapping is not None
            else regime_id
        )

        rows.append({
            "regime_id": regime_id,
            "regime_name": regime_name,
            "start_minute_of_day": start_min,
            "end_minute_of_day": end_min,
            "start_hhmm": f"{start_min // 60:02d}:{start_min % 60:02d}",
            "end_hhmm": f"{end_min // 60:02d}:{end_min % 60:02d}",
            "n_rows": int(m.shape[0]),
        })

    out = pd.DataFrame(rows)

    out = out.sort_values(
        by="start_minute_of_day"
    ).reset_index(drop=True)

    return out[
        [
            "regime_id",
            "regime_name",
            "start_minute_of_day",
            "end_minute_of_day",
            "start_hhmm",
            "end_hhmm",
            "n_rows",
        ]
    ]

In [71]:
regime_mapping = {
    0: "overnight",
    1: "premarket",
    2: "opening",
    3: "regular",
    4: "closing",
}

regime_table = build_regime_ranges_table(
    mnq_intraday_t2_ti,
    regime_mapping=regime_mapping,
)

regime_table

,regime_id,regime_name,start_minute_of_day,end_minute_of_day,start_hhmm,end_hhmm,n_rows
0,0,overnight,359,960,05:59,16:00,225264
1,1,premarket,510,569,08:30,09:29,88920
2,2,opening,570,629,09:30,10:29,88920
3,3,regular,630,929,10:30,15:29,444600
4,4,closing,930,959,15:30,15:59,44460


### **4.2.2. Funciones para calcular IC**

In [72]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return spearmanr(x[mask], y[mask]).correlation


def filter_market_regimes(
    df: pd.DataFrame,
    regime_ids: list[int] | tuple[int, ...] | None = None,
    regime_col: str = "regime_id",
) -> pd.DataFrame:
    """
    Filtra el DataFrame por uno o varios regímenes.

    Parámetros
    ----------
    regime_ids : list[int] | tuple[int, ...] | None
        - None -> no filtra
        - [1, 2] -> usa solo filas con esos regímenes
    """
    if regime_ids is None:
        return df.copy()

    if regime_col not in df.columns:
        raise KeyError(f"Falta la columna '{regime_col}' en el DataFrame.")

    return df.loc[df[regime_col].isin(regime_ids)].copy()


def daily_ic(
    df: pd.DataFrame,
    indicator_col: str,
    target_col: str,
    date_col: str = "date",
) -> pd.Series:
    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")

    return df.groupby(date_col).apply(
        lambda g: spearman_ic(g[indicator_col], g[target_col])
    )


def extract_horizon_from_target(target_col: str) -> int | float:
    """
    Extrae el horizonte desde nombres como:
    - t2_p40_h30
    - t2_p50_h60
    """
    try:
        last_part = target_col.split("_")[-1]   # ej: h30
        return int(last_part.replace("h", ""))
    except Exception:
        return np.nan


def compute_ic_table_is_oos_t2(
    df: pd.DataFrame,
    indicator_columns: list[str],
    target_columns: list[str],
    *,
    regime_ids: list[int] | tuple[int, ...] | None = None,
    regime_col: str = "regime_id",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Calcula IC Spearman entre indicadores y targets T2,
    separando IS/OOS y filtrando por uno o varios regímenes.

    Devuelve una tabla con:
      - indicator
      - target_col
      - horizon
      - regime_scope
      - IC_IS
      - IC_OOS
      - oos_minus_is
      - n_pairs_IS
      - n_pairs_OOS
      - abs_IC_OOS
      - note
    """

    # 1) Filtrado por regímenes
    dfr = filter_market_regimes(df, regime_ids=regime_ids, regime_col=regime_col)

    # 2) Validaciones estructurales
    required = {split_col, date_col}
    missing_req = [c for c in required if c not in dfr.columns]
    if missing_req:
        raise ValueError(f"Faltan columnas requeridas: {missing_req}")

    # 3) Subsets IS / OOS
    df_is = dfr[dfr[split_col] == "IS"].copy()
    df_oos = dfr[dfr[split_col] == "OOS"].copy()

    regime_scope = "full_day" if regime_ids is None else "_".join(map(str, regime_ids))

    rows = []

    for ind in indicator_columns:
        for target_col in target_columns:

            horizon = extract_horizon_from_target(target_col)

            missing = [c for c in (ind, target_col) if c not in dfr.columns]
            if missing:
                rows.append({
                    "indicator": ind,
                    "target_col": target_col,
                    "horizon": horizon,
                    "regime_scope": regime_scope,
                    "IC_IS": np.nan,
                    "IC_OOS": np.nan,
                    "oos_minus_is": np.nan,
                    "n_pairs_IS": 0,
                    "n_pairs_OOS": 0,
                    "abs_IC_OOS": np.nan,
                    "note": f"missing: {missing}",
                })
                continue

            n_pairs_is = int((df_is[ind].notna() & df_is[target_col].notna()).sum())
            n_pairs_oos = int((df_oos[ind].notna() & df_oos[target_col].notna()).sum())

            # IS
            if use_daily_ic:
                ic_is_series = daily_ic(df_is, ind, target_col, date_col=date_col)
                ic_is = float(ic_is_series.mean()) if len(ic_is_series) > 0 else np.nan
            else:
                ic_is = float(spearman_ic(df_is[ind], df_is[target_col]))

            # OOS
            if use_daily_ic:
                ic_oos_series = daily_ic(df_oos, ind, target_col, date_col=date_col)
                ic_oos = float(ic_oos_series.mean()) if len(ic_oos_series) > 0 else np.nan
            else:
                ic_oos = float(spearman_ic(df_oos[ind], df_oos[target_col]))

            rows.append({
                "indicator": ind,
                "target_col": target_col,
                "horizon": horizon,
                "regime_scope": regime_scope,
                "IC_IS": ic_is,
                "IC_OOS": ic_oos,
                "oos_minus_is": (ic_oos - ic_is) if pd.notna(ic_is) and pd.notna(ic_oos) else np.nan,
                "n_pairs_IS": n_pairs_is,
                "n_pairs_OOS": n_pairs_oos,
                "abs_IC_OOS": abs(ic_oos) if pd.notna(ic_oos) else np.nan,
                "note": "",
            })

    out = pd.DataFrame(rows)

    out = (
        out.sort_values(
            ["regime_scope", "target_col", "abs_IC_OOS"],
            ascending=[True, True, False]
        )
        .reset_index(drop=True)
    )

    return out

In [73]:
target_cols_t2 = [
        "t2_p40_h30",
        "t2_p40_h60",
        "t2_p50_h30",
   ]

### **4.2.3. Función para calculo de IC tables**

In [74]:
import os
import json
import pandas as pd

# ============================================================
# Cache de IC tables (T2)
# ============================================================

CACHE_DIR_T2 = "/content/drive/MyDrive/neural_profit/data/03_targets/ic_tables_t2"
os.makedirs(CACHE_DIR_T2, exist_ok=True)


def _select_cache_dir() -> str:
    return CACHE_DIR_T2


def _ic_cache_paths(cache_dir: str, name: str) -> dict:
    return {
        "data": os.path.join(cache_dir, f"{name}.parquet"),
        "meta": os.path.join(cache_dir, f"{name}.meta.json"),
    }


def load_or_compute_ic_table_t2(
    *,
    name: str,
    df: pd.DataFrame,
    indicator_columns: list[str],
    target_columns: list[str],
    regime_ids: list[int] | tuple[int, ...] | None = None,
    regime_col: str = "regime_id",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Carga una ic_table T2 desde cache si existe; si no, la calcula y la guarda.
    """
    cache_dir = _select_cache_dir()
    paths = _ic_cache_paths(cache_dir, name)

    # 1) Cargar desde cache
    if (not force_recompute) and os.path.exists(paths["data"]):
        ic_table = pd.read_parquet(paths["data"])
        print(f"[cache] Loaded: {paths['data']}")
        return ic_table

    # 2) Calcular ic_table
    ic_table = compute_ic_table_is_oos_t2(
        df=df,
        indicator_columns=indicator_columns,
        target_columns=target_columns,
        regime_ids=regime_ids,
        regime_col=regime_col,
        use_daily_ic=use_daily_ic,
        split_col=split_col,
        date_col=date_col,
    )

    # 3) Guardar datos
    ic_table.to_parquet(paths["data"], index=False)

    # 4) Guardar metadatos
    meta = {
        "name": name,
        "target_type": "t2",
        "target_columns": list(target_columns),
        "regime_ids": list(regime_ids) if regime_ids is not None else "full_day",
        "regime_col": regime_col,
        "use_daily_ic": use_daily_ic,
        "split_col": split_col,
        "date_col": date_col,
        "n_indicators": len(indicator_columns),
        "cache_dir": cache_dir,
    }

    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[cache] Computed & saved: {paths['data']}")
    return ic_table

### **4.2.4. Aplicación de cálculos**

In [75]:
def build_selected_ic_tables_t2(
    df: pd.DataFrame,
    *,
    indicator_columns: list[str],
    target_columns: list[str],
    regime_col: str = "regime_id",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
    force_recompute: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Calcula o carga solo las tablas IC relevantes para T2
    en los regímenes de interés.

    Tablas:
    - regimes_1_2
    - regime_1
    - regime_2
    """

    if regime_col not in df.columns:
        raise KeyError(f"Falta la columna '{regime_col}' en el DataFrame.")

    ic_tables = {
        "regimes_1_2": load_or_compute_ic_table_t2(
            name="ic_table_t2_regimes_1_2",
            df=df,
            indicator_columns=indicator_columns,
            target_columns=target_columns,
            regime_ids=[1, 2],
            regime_col=regime_col,
            use_daily_ic=use_daily_ic,
            split_col=split_col,
            date_col=date_col,
            force_recompute=force_recompute,
        ),
        "regime_1": load_or_compute_ic_table_t2(
            name="ic_table_t2_regime_1",
            df=df,
            indicator_columns=indicator_columns,
            target_columns=target_columns,
            regime_ids=[1],
            regime_col=regime_col,
            use_daily_ic=use_daily_ic,
            split_col=split_col,
            date_col=date_col,
            force_recompute=force_recompute,
        ),
        "regime_2": load_or_compute_ic_table_t2(
            name="ic_table_t2_regime_2",
            df=df,
            indicator_columns=indicator_columns,
            target_columns=target_columns,
            regime_ids=[2],
            regime_col=regime_col,
            use_daily_ic=use_daily_ic,
            split_col=split_col,
            date_col=date_col,
            force_recompute=force_recompute,
        ),
    }

    return ic_tables

In [78]:
target_cols_t2 = [
        "t2_p40_h30",
        "t2_p40_h60",
        "t2_p50_h30",
   ]

ic_tables_t2 = build_selected_ic_tables_t2(
    df=mnq_intraday_t2_ti,
    indicator_columns=ti_cols,
    target_columns=target_cols_t2,
    regime_col="regime_id",
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
    force_recompute=False,
)

[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/03_targets/ic_tables_t2/ic_table_t2_regimes_1_2.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/03_targets/ic_tables_t2/ic_table_t2_regime_1.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/03_targets/ic_tables_t2/ic_table_t2_regime_2.parquet


In [79]:
ic_tables_t2.keys()

dict_keys(['regimes_1_2', 'regime_1', 'regime_2'])

In [80]:
regime_name_map = {
    0: "overnight",
    1: "premarket",
    2: "opening",
    3: "regular",
    4: "closing",
}

In [83]:
ic_table_premarket_opening = ic_tables_t2["regimes_1_2"]
ic_table_premarket  = ic_tables_t2["regime_1"]
ic_table_opening    = ic_tables_t2["regime_2"]


In [84]:
ic_table_premarket_opening

,indicator,target_col,horizon,regime_scope,IC_IS,IC_OOS,oos_minus_is,n_pairs_IS,n_pairs_OOS,abs_IC_OOS,note
0,roc_60,t2_p40_h30,30,1_2,-0.301845,-0.295472,0.006373,113520,64320,0.295472,
1,ema_60,t2_p40_h30,30,1_2,-0.261883,-0.246606,0.015277,113520,64320,0.246606,
2,roc_30,t2_p40_h30,30,1_2,-0.211742,-0.201806,0.009936,113520,64320,0.201806,
3,ema_30,t2_p40_h30,30,1_2,-0.204020,-0.184025,0.019995,113520,64320,0.184025,
4,bb_60_15,t2_p40_h30,30,1_2,-0.191306,-0.178910,0.012396,113520,64320,0.178910,
...,...,...,...,...,...,...,...,...,...,...,...
121,volume_ratio_90,t2_p50_h30,30,1_2,0.011280,0.029627,0.018347,113520,64320,0.029627,
122,volume_ratio_60,t2_p50_h30,30,1_2,0.013190,0.023197,0.010006,113520,64320,0.023197,
123,volume_ratio_30,t2_p50_h30,30,1_2,0.014856,0.014375,-0.000480,113520,64320,0.014375,
124,volume_ratio_20,t2_p50_h30,30,1_2,0.012786,0.013992,0.001206,113520,64320,0.013992,


In [85]:
target_cols_t2 = [
        "t2_p40_h30",
        "t2_p40_h60",
        "t2_p50_h30",
   ]

def clean_ic_table(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df[
            df["note"].eq("") &
            df["IC_OOS"].notna()
        ]
        .copy()
    )

ic_table_premarket_opening = clean_ic_table(ic_table_premarket_opening)
ic_table_premarket = clean_ic_table(ic_table_premarket)
ic_table_opening = clean_ic_table(ic_table_opening)

## **4.3. Análisis de resultados**

Qué vamos a analizar primero:

- Magnitud del IC_OOS → ¿hay señal?
- Signo del IC_OOS → ¿momentum o reversión?
- Top indicadores por régimen → ¿se repiten o cambian?

Sirve para responder rápido cosas como:

- ¿en qué régimen hay más señal OOS?
- ¿predominan relaciones positivas o negativas?
- ¿cuáles son los indicadores más fuertes en cada régimen?

Qué no hace
No te dice todavía:
- si el indicador es robusto IS vs OOS
- si hay redundancia entre indicadores
- si un indicador sirve en varios regímenes

O sea: sirve para exploración inicial, no para selección final.

### **Código**


In [87]:
import pandas as pd


def quick_ic_observation_t2(
    ic_tables: dict[str, pd.DataFrame],
    *,
    target_col: str,
    top_n: int = 5,
) -> None:
    """
    Resumen exploratorio rápido de tablas IC para un target T2 específico.

    Para un target dado:
    - muestra top indicadores por |IC_OOS|
    - resume signo predominante
    - resume magnitud promedio de IC_OOS

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Diccionario de tablas IC, por ejemplo:
        {
            "regimes_1_2": ...,
            "regime_1": ...,
            "regime_2": ...,
        }
    target_col : str
        Target específico a analizar, por ejemplo:
        "t2_p40_h30"
    top_n : int
        Número de indicadores top a mostrar por tabla.
    """

    print("\n" + "=" * 90)
    print(f"ANÁLISIS SIMPLE IC | target={target_col}")
    print("=" * 90)

    for regime_name, df in ic_tables.items():
        x = df[df["target_col"] == target_col].copy()
        x = x[x["IC_OOS"].notna()].copy()
        x = x[x["note"].fillna("") == ""].copy()

        if x.empty:
            print("\n" + "-" * 90)
            print(f"Régimen: {regime_name}")
            print("-" * 90)
            print("Sin resultados válidos para este target.")
            continue

        x["abs_IC_OOS"] = x["IC_OOS"].abs()

        top = x.sort_values("abs_IC_OOS", ascending=False).head(top_n)

        mean_ic = x["IC_OOS"].mean()
        mean_abs_ic = x["abs_IC_OOS"].mean()

        pct_positive = (x["IC_OOS"] > 0).mean() * 100
        pct_negative = (x["IC_OOS"] < 0).mean() * 100
        pct_zero = (x["IC_OOS"] == 0).mean() * 100

        print("\n" + "-" * 90)
        print(f"Régimen: {regime_name}")
        print("-" * 90)

        print(f"IC_OOS medio    : {mean_ic:.4f}")
        print(f"|IC_OOS| medio  : {mean_abs_ic:.4f}")
        print(f"% IC positivo   : {pct_positive:.1f}%")
        print(f"% IC negativo   : {pct_negative:.1f}%")
        print(f"% IC cero       : {pct_zero:.1f}%")

        print("\nTop indicadores por |IC_OOS|:")
        print(
            top[
                [
                    "indicator",
                    "target_col",
                    "IC_OOS",
                    "abs_IC_OOS",
                ]
            ].to_string(index=False)
        )

In [88]:
target_cols_t2

['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']

In [89]:
quick_ic_observation_t2(
    ic_tables_t2,
    target_col="t2_p40_h30",
    top_n=10,
)


ANÁLISIS SIMPLE IC | target=t2_p40_h30

------------------------------------------------------------------------------------------
Régimen: regimes_1_2
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.0891
|IC_OOS| medio  : 0.1125
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
 indicator target_col    IC_OOS  abs_IC_OOS
    roc_60 t2_p40_h30 -0.295472    0.295472
    ema_60 t2_p40_h30 -0.246606    0.246606
    roc_30 t2_p40_h30 -0.201806    0.201806
    ema_30 t2_p40_h30 -0.184025    0.184025
  bb_60_15 t2_p40_h30 -0.178910    0.178910
  bb_60_20 t2_p40_h30 -0.178910    0.178910
  bb_60_25 t2_p40_h30 -0.178910    0.178910
    rsi_14 t2_p40_h30 -0.169070    0.169070
    roc_20 t2_p40_h30 -0.153731    0.153731
stoch_k_30 t2_p40_h30 -0.151437    0.151437

------------------------------------------------------------------------------------------
Régimen: regime_1
---------

In [90]:
quick_ic_observation_t2(
    ic_tables_t2,
    target_col="t2_p40_h60",
    top_n=10,
)


ANÁLISIS SIMPLE IC | target=t2_p40_h60

------------------------------------------------------------------------------------------
Régimen: regimes_1_2
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.1383
|IC_OOS| medio  : 0.1588
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
indicator target_col    IC_OOS  abs_IC_OOS
   roc_60 t2_p40_h60 -0.377783    0.377783
   ema_60 t2_p40_h60 -0.343554    0.343554
   roc_30 t2_p40_h60 -0.285733    0.285733
   ema_30 t2_p40_h60 -0.274538    0.274538
 bb_60_15 t2_p40_h60 -0.259908    0.259908
 bb_60_20 t2_p40_h60 -0.259908    0.259908
 bb_60_25 t2_p40_h60 -0.259908    0.259908
   rsi_14 t2_p40_h60 -0.239124    0.239124
   roc_20 t2_p40_h60 -0.233645    0.233645
   ema_20 t2_p40_h60 -0.230908    0.230908

------------------------------------------------------------------------------------------
Régimen: regime_1
--------------------

In [91]:
quick_ic_observation_t2(
    ic_tables_t2,
    target_col='t2_p50_h30',
    top_n=10,
)


ANÁLISIS SIMPLE IC | target=t2_p50_h30

------------------------------------------------------------------------------------------
Régimen: regimes_1_2
------------------------------------------------------------------------------------------
IC_OOS medio    : -0.0882
|IC_OOS| medio  : 0.1118
% IC positivo   : 23.8%
% IC negativo   : 76.2%
% IC cero       : 0.0%

Top indicadores por |IC_OOS|:
 indicator target_col    IC_OOS  abs_IC_OOS
    roc_60 t2_p50_h30 -0.289254    0.289254
    ema_60 t2_p50_h30 -0.242407    0.242407
    roc_30 t2_p50_h30 -0.200467    0.200467
    ema_30 t2_p50_h30 -0.183655    0.183655
  bb_60_15 t2_p50_h30 -0.174656    0.174656
  bb_60_20 t2_p50_h30 -0.174656    0.174656
  bb_60_25 t2_p50_h30 -0.174656    0.174656
    rsi_14 t2_p50_h30 -0.166187    0.166187
    roc_20 t2_p50_h30 -0.155584    0.155584
stoch_k_30 t2_p50_h30 -0.151470    0.151470

------------------------------------------------------------------------------------------
Régimen: regime_1
---------

### **Observaciones**


**1. La señal es consistente entre los tres targets**

* Magnitudes de IC muy similares entre:

  * `t2_p40_h30`
  * `t2_p40_h60`
  * `t2_p50_h30`
* El comportamiento no cambia de forma significativa al modificar:

  * horizonte (30 → 60)
  * percentil (p40 → p50)

Esto sugiere que la señal capturada es estructural.

---

**2. Predomina claramente la reversión a la media**

* ~76% de los IC son negativos en todos los casos
* El signo es estable entre targets y regímenes

Conclusión: la dinámica dominante es consistente y no depende del target específico.

---

**3. El régimen 2 concentra la mayor calidad de señal**

* Siempre presenta:

  * mayor |IC_OOS|
  * mayores valores extremos (≈ 0.54)
* Se mantiene como el entorno más informativo en los tres targets

---

**4. Los indicadores líderes son prácticamente idénticos**

Indicadores que aparecen en el top en los tres targets:

* `roc_60`
* `ema_60`
* `roc_30`
* `ema_30`
* `rsi_14`
* `stoch_k_30`
* `roc_20`
* `ema_20`
* `bb_60_*`

Esto indica:

* alta consistencia entre targets
* la señal no depende de una configuración puntual

---

**5. Dominio de indicadores de tendencia y momentum**

Los indicadores más relevantes pertenecen a:

* retornos (ROC)
* medias móviles (EMA)
* osciladores (RSI, Stochastic)
* bandas de Bollinger

Interpretación:

* el mercado muestra patrones de sobre-extensión y reversión
* las features capturan correctamente esa dinámica

---

**6. Escalamiento con el horizonte**

* Al pasar de 30 → 60 minutos:

  * aumenta la magnitud del IC
* Esto sugiere que:

  * la reversión se materializa mejor en horizontes un poco más largos

---

**7. El cambio de percentil (p40 → p50) no altera la estructura**

* `t2_p40_h30` vs `t2_p50_h30`:

  * mismos indicadores
  * magnitudes muy similares
* Implica que:

  * la señal es robusta al threshold elegido

---

**Conclusión**

* Existe un conjunto claro de indicadores que explican consistentemente el comportamiento de los tres targets
* La señal es estable en:

  * signo (negativa)
  * magnitud
  * ranking de features
* Los indicadores clave están ligados a momentum y tendencia, pero capturan un régimen de reversión
* El régimen 2 es el entorno más informativo
* El comportamiento es robusto frente a cambios de horizonte y percentil, lo que refuerza la validez de estas features como candidatas iniciales para selección posterior


# **5. Análisis de robustez IS vs OOS**


El objetivo de este análisis es evaluar la estabilidad de la señal de los indicadores técnicos al pasar de datos in-sample (IS) a out-of-sample (OOS).

En el contexto de machine learning aplicado a trading, este punto es crítico, ya que un indicador puede mostrar una relación aparente fuerte en entrenamiento, pero perder completamente su capacidad predictiva al generalizar. Este comportamiento es característico del sobreajuste (overfitting), y es precisamente lo que se busca detectar en esta etapa.

---

Para ello, se compara el Information Coefficient (IC) calculado en IS y en OOS para cada indicador, horizonte y régimen de mercado.

La lógica del análisis se basa en los siguientes criterios:

- Diferencia entre IC_IS e IC_OOS (gap)
  
  Se define como:
  
  gap = IC_OOS - IC_IS

  Interpretación:
  
  - gap pequeño → la señal se mantiene al pasar a OOS (mayor robustez)
  - gap grande → la señal cambia significativamente (posible inestabilidad)

---

- Magnitud del gap (abs_gap)

  abs_gap = |IC_OOS - IC_IS|

  Este valor permite medir directamente la estabilidad de la señal sin considerar el signo.

  Interpretación:
  
  - abs_gap bajo → indicador estable
  - abs_gap alto → indicador inestable

---

- Consistencia de signo entre IS y OOS

  Se evalúa si el signo del IC se mantiene:

  - mismo signo → la relación entre indicador y target se conserva
  - cambio de signo → la señal es inconsistente o potencialmente espuria

  Este criterio es especialmente importante en targets T2, ya que el signo del IC define si el indicador está asociado a movimientos positivos o negativos significativos.

---

- IC_OOS como referencia principal

  Aunque se analiza la relación entre IS y OOS, el valor más importante es IC_OOS, ya que representa la capacidad predictiva fuera de muestra.

  Por lo tanto:

  - IC_OOS alto + bajo abs_gap → indicador robusto
  - IC_OOS alto + alto abs_gap → indicador sospechoso
  - IC_OOS bajo → indicador poco útil, independientemente de su estabilidad

---

Este análisis permite:

- identificar indicadores que mantienen su señal fuera de muestra,
- descartar indicadores que dependen del dataset de entrenamiento,
- priorizar factores robustos y generalizables,
- reducir el riesgo de sobreajuste en etapas posteriores de modelado.

---

En conjunto, la robustez IS vs OOS constituye un criterio fundamental para la selección de features en problemas financieros, donde la capacidad de generalización es más importante que el ajuste en entrenamiento.

## **5.1. Implementación de código**

In [92]:
import numpy as np
import pandas as pd

def analyze_ic_robustness(
    ic_table: pd.DataFrame,
    *,
    regime_name: str,
    min_abs_ic_oos: float = 0.05,
    require_same_sign: bool = True,
    top_n: int = 10,
) -> dict[str, pd.DataFrame]:
    """
    Analiza robustez IS vs OOS para una tabla IC de un régimen.

    Devuelve:
    ---------
    {
        "summary_df": resumen por horizonte,
        "top_df": top indicadores robustos por horizonte
    }
    """

    required = [
        "indicator", "horizon", "target_col",
        "IC_IS", "IC_OOS"
    ]
    missing = [c for c in required if c not in ic_table.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    df = ic_table.copy()
    df = df[df["IC_IS"].notna() & df["IC_OOS"].notna()].copy()

    if df.empty:
        empty_summary = pd.DataFrame([{
            "regime_name": regime_name,
            "horizon": pd.NA,
            "n_total": 0,
            "n_after_filter": 0,
            "pct_retained": np.nan,
            "mean_IC_IS": np.nan,
            "mean_IC_OOS": np.nan,
            "mean_abs_IC_OOS": np.nan,
            "mean_abs_gap": np.nan,
            "pct_same_sign": np.nan,
            "best_indicator": pd.NA,
            "best_IC_OOS": np.nan,
            "best_abs_gap": np.nan,
            "best_robustness_score": np.nan,
        }])
        empty_top = pd.DataFrame(columns=[
            "regime_name", "indicator", "horizon", "target_col",
            "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
            "same_sign", "abs_IC_OOS", "strength_ratio",
            "robustness_score"
        ])
        return {"summary_df": empty_summary, "top_df": empty_top}

    # Variables base
    df["oos_minus_is"] = df["IC_OOS"] - df["IC_IS"]
    df["abs_gap"] = df["oos_minus_is"].abs()
    df["same_sign"] = np.sign(df["IC_IS"]) == np.sign(df["IC_OOS"])
    df["abs_IC_IS"] = df["IC_IS"].abs()
    df["abs_IC_OOS"] = df["IC_OOS"].abs()

    df["strength_ratio"] = np.where(
        df["abs_IC_IS"] > 1e-12,
        df["abs_IC_OOS"] / df["abs_IC_IS"],
        np.nan,
    )

    sign_bonus = np.where(df["same_sign"], 1.0, 0.0)

    # Score robustez
    df["robustness_score"] = (
        0.70 * df["abs_IC_OOS"]
        - 0.25 * df["abs_gap"]
        + 0.05 * sign_bonus
    )

    # Conteo previo por horizonte
    n_total_by_h = (
        df.groupby("horizon")
          .size()
          .rename("n_total")
          .reset_index()
    )

    # Filtro mínimo de señal
    df = df[df["abs_IC_OOS"] >= min_abs_ic_oos].copy()

    # Filtro opcional por consistencia de signo
    if require_same_sign:
        df = df[df["same_sign"]].copy()

    # Si tras filtrar queda vacío
    if df.empty:
        summary_rows = []
        for _, row in n_total_by_h.iterrows():
            summary_rows.append({
                "regime_name": regime_name,
                "horizon": row["horizon"],
                "n_total": int(row["n_total"]),
                "n_after_filter": 0,
                "pct_retained": 0.0,
                "mean_IC_IS": np.nan,
                "mean_IC_OOS": np.nan,
                "mean_abs_IC_OOS": np.nan,
                "mean_abs_gap": np.nan,
                "pct_same_sign": np.nan,
                "best_indicator": pd.NA,
                "best_IC_OOS": np.nan,
                "best_abs_gap": np.nan,
                "best_robustness_score": np.nan,
            })
        return {
            "summary_df": pd.DataFrame(summary_rows),
            "top_df": pd.DataFrame(columns=[
                "regime_name", "indicator", "horizon", "target_col",
                "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
                "same_sign", "abs_IC_OOS", "strength_ratio",
                "robustness_score"
            ])
        }

    # Orden final
    df = df.sort_values(
        ["horizon", "robustness_score", "abs_IC_OOS", "abs_gap"],
        ascending=[True, False, False, True]
    ).reset_index(drop=True)

    df["regime_name"] = regime_name

    # Top robustos por horizonte
    top_df = (
        df.groupby("horizon", group_keys=False)
          .head(top_n)
          .reset_index(drop=True)
    )[
        [
            "regime_name", "indicator", "horizon", "target_col",
            "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
            "same_sign", "abs_IC_OOS", "strength_ratio",
            "robustness_score"
        ]
    ]

    # Resumen exacto para conclusiones
    summary_rows = []
    for h, g in df.groupby("horizon"):
        n_total = int(n_total_by_h.loc[n_total_by_h["horizon"] == h, "n_total"].iloc[0])
        best = g.iloc[0]

        summary_rows.append({
            "regime_name": regime_name,
            "horizon": h,
            "n_total": n_total,
            "n_after_filter": int(len(g)),
            "pct_retained": float(len(g) / n_total * 100) if n_total > 0 else np.nan,
            "mean_IC_IS": float(g["IC_IS"].mean()),
            "mean_IC_OOS": float(g["IC_OOS"].mean()),
            "mean_abs_IC_OOS": float(g["abs_IC_OOS"].mean()),
            "mean_abs_gap": float(g["abs_gap"].mean()),
            "pct_same_sign": float(g["same_sign"].mean() * 100),
            "best_indicator": best["indicator"],
            "best_IC_OOS": float(best["IC_OOS"]),
            "best_abs_gap": float(best["abs_gap"]),
            "best_robustness_score": float(best["robustness_score"]),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("horizon").reset_index(drop=True)

    return {
        "summary_df": summary_df,
        "top_df": top_df,
    }

In [93]:
rob_premarket_opening = analyze_ic_robustness(ic_table_premarket_opening, regime_name="premarket_opening")
rob_premarket = analyze_ic_robustness(ic_table_premarket, regime_name="premarket")
rob_opening   = analyze_ic_robustness(ic_table_opening,   regime_name="opening")

## **5.2. Summary all**

In [94]:
robustness_summary_all = pd.concat([
    rob_premarket_opening["summary_df"],
    rob_premarket["summary_df"],
    rob_opening["summary_df"],
], ignore_index=True)

robustness_summary_all


,regime_name,horizon,n_total,n_after_filter,pct_retained,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,best_indicator,best_IC_OOS,best_abs_gap,best_robustness_score
0,premarket_opening,30,84,72,85.714286,-0.125560,-0.104750,0.127375,0.020810,100.0,roc_60,-0.295472,0.006373,0.255238
1,premarket_opening,60,42,37,88.095238,-0.175686,-0.159408,0.177799,0.016277,100.0,roc_60,-0.377783,0.000988,0.314201
2,premarket,30,84,74,88.095238,-0.114759,-0.134772,0.151962,0.034471,100.0,roc_60,-0.311729,0.075117,0.249431
3,premarket,60,42,32,76.190476,-0.133279,-0.148763,0.148763,0.015721,100.0,roc_60,-0.238277,0.042374,0.206200
4,opening,30,84,74,88.095238,-0.229971,-0.219376,0.256026,0.011440,100.0,roc_60,-0.545564,0.007568,0.430003
5,opening,60,42,38,90.476190,-0.236551,-0.214096,0.252453,0.023195,100.0,roc_60,-0.545866,0.014061,0.428591


### **Conclusiones — Análisis de robustez IS vs OOS para targets T2**


**1. Alta estabilidad IS vs OOS**

* `mean_abs_gap` bajo en todos los casos (~0.01 – 0.03)
* Indica que la magnitud del IC se mantiene al pasar de IS a OOS

Esto sugiere ausencia de sobreajuste significativo.

---

**2. Consistencia total de signo**

* `pct_same_sign = 100%` en todos los regímenes y horizontes
* La relación entre indicadores y target se mantiene sin cambios

Esto es una señal muy fuerte de robustez estructural.

---

**3. Buena retención de indicadores**

* `pct_retained` entre ~76% y 90%
* La mayoría de los indicadores mantienen comportamiento válido en OOS

Implica que la señal no depende de un subconjunto reducido.

---

**4. IC_OOS significativo y consistente**

* `mean_abs_IC_OOS`:

  * premarket_opening: ~0.13 – 0.18
  * premarket: ~0.15
  * opening: ~0.25

El régimen `opening` presenta la mayor calidad de señal.

---

**5. Régimen opening claramente superior**

* Mayor `mean_abs_IC_OOS` (~0.25)
* Menor `mean_abs_gap` (~0.01 – 0.02)
* Mejores `best_IC_OOS` (~0.54)

Es el entorno más robusto y explotable.

---

**6. Indicador dominante consistente**

* `roc_60` es el mejor indicador en todos los casos
* Aparece como `best_indicator` en todos los regímenes y horizontes

Esto confirma lo observado en el análisis exploratorio previo.

---

**7. Mejora al aumentar horizonte**

* De 30 → 60 minutos:

  * aumenta |IC_OOS|
  * mejora robustness_score

Sugiere que la señal se expresa mejor en horizontes más largos.

---

**8. Robustness score elevado**

* Valores entre ~0.20 y 0.43
* Máximos en régimen opening

Indica combinación favorable de:

* alta señal OOS
* baja degradación respecto a IS

---

**Conclusión**

* Los indicadores presentan **alta robustez temporal**
* La señal se mantiene estable entre IS y OOS tanto en magnitud como en signo
* El régimen `opening` concentra la mejor calidad y estabilidad
* `roc_60` emerge como el indicador más consistente y fuerte
* No hay evidencia clara de sobreajuste en esta etapa
* El conjunto de features es adecuado para avanzar hacia selección final y modelado


## **5.3. Top indicadores robustos**

In [95]:
robustness_top_all = pd.concat([
    rob_premarket_opening["top_df"],
    rob_premarket["top_df"],
    rob_opening["top_df"],

], ignore_index=True)

#robustness_top_all

def find_common_indicators(robustness_top_all: pd.DataFrame) -> pd.DataFrame:
    df = robustness_top_all.copy()

    # Conteo de aparición por indicador
    summary = (
        df.groupby("indicator")
        .agg(
            n_regimes=("regime_name", "nunique"),
            regimes=("regime_name", lambda x: sorted(set(x))),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_score=("robustness_score", "mean"),
        )
        .reset_index()
        .sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])
    )

    return summary

common_indicators = find_common_indicators(robustness_top_all)
common_indicators

,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_score
8,roc_60,3,"[opening, premarket, premarket_opening]",-0.379127,0.379127,0.308956
5,ema_60,3,"[opening, premarket, premarket_opening]",-0.335433,0.335433,0.278680
7,roc_30,3,"[opening, premarket, premarket_opening]",-0.286109,0.286109,0.246074
4,ema_30,3,"[opening, premarket, premarket_opening]",-0.274497,0.274497,0.237001
9,rsi_14,3,"[opening, premarket, premarket_opening]",-0.268324,0.268324,0.232261
3,ema_20,3,"[opening, premarket, premarket_opening]",-0.238316,0.238316,0.212002
0,bb_60_15,3,"[opening, premarket, premarket_opening]",-0.228657,0.228657,0.204124
1,bb_60_20,3,"[opening, premarket, premarket_opening]",-0.228657,0.228657,0.204124
10,stoch_k_30,2,"[opening, premarket]",-0.286405,0.286405,0.244298
6,roc_20,2,"[opening, premarket_opening]",-0.277167,0.277167,0.237909


### **Conclusiones — Indicadores técnicos más robustos (análisis cross-regime)**

**1. Existe un núcleo claro de indicadores robustos**

Indicadores presentes en los tres regímenes (`n_regimes = 3`):

* `roc_60`
* `ema_60`
* `roc_30`
* `ema_30`
* `rsi_14`
* `ema_20`
* `bb_60_15`
* `bb_60_20`

Esto indica que estos indicadores:

* son consistentes entre regímenes
* mantienen señal OOS
* no dependen de un contexto específico

---

**2. Liderazgo claro de ROC y EMA**

* `roc_60` es el mejor indicador global
* seguido por `ema_60`, `roc_30`, `ema_30`

Interpretación:

* la información relevante está en:

  * retornos recientes
  * suavizados de tendencia

---

**3. Magnitudes de IC consistentes y relevantes**

* `mean_abs_IC_OOS` entre ~0.22 y 0.38
* valores elevados para un problema financiero

Esto refuerza que:

* la señal es fuerte
* y estable entre regímenes

---

**4. Señal coherente en signo**

* todos los IC son negativos
* consistente con el análisis previo

Conclusión:

* los indicadores capturan dinámica de reversión

---

**5. Indicadores de segundo nivel**

Indicadores presentes en 2 regímenes:

* `stoch_k_30`
* `roc_20`
* `bb_60_25`

Interpretación:

* tienen señal, pero menos robusta
* pueden ser útiles como complemento

---

**6. Estructura clara del set de features**

Se identifican tres grupos:

* Núcleo robusto (n_regimes = 3) → candidatos principales
* Intermedios (n_regimes = 2) → candidatos secundarios
* Resto → probablemente descartables

---

**7. Redundancia implícita**

* `roc_60`, `roc_30`, `roc_20` → misma familia
* `ema_60`, `ema_30`, `ema_20` → misma familia
* `bb_60_*` → misma lógica

Esto sugiere:

* alta correlación entre features
* necesidad posterior de reducción de redundancia

---

**Conclusión**

* Existe un conjunto compacto de indicadores altamente robustos
* La señal es consistente entre regímenes y targets
* ROC y EMA dominan el comportamiento predictivo
* La estructura es coherente con una dinámica de reversión
* El siguiente paso debe enfocarse en:

  * eliminar redundancia
  * seleccionar subconjunto óptimo de features


# **6. Consistencia de signo**

## **6.1. Función General**

In [96]:
import numpy as np
import pandas as pd

def analyze_sign_consistency_across_regimes(
    ic_tables: dict[str, pd.DataFrame],
    *,
    top_n_inconsistent: int = 10,
) -> dict[str, pd.DataFrame]:
    """
    Analiza la consistencia de signo entre IC_IS e IC_OOS
    para múltiples tablas IC por régimen.

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Diccionario tipo:
        {
            "overnight": ic_table_overnight,
            "premarket": ic_table_premarket,
            ...
        }

    Devuelve
    --------
    {
        "summary_df": resumen por régimen y horizonte,
        "inconsistent_df": indicadores que cambian de signo
    }
    """

    all_rows = []

    for regime_name, df in ic_tables.items():
        required = ["indicator", "horizon", "target_col", "IC_IS", "IC_OOS"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"En régimen '{regime_name}' faltan columnas: {missing}")

        x = df.copy()
        x = x[x["IC_IS"].notna() & x["IC_OOS"].notna()].copy()

        if x.empty:
            continue

        x["regime_name"] = regime_name
        x["same_sign"] = np.sign(x["IC_IS"]) == np.sign(x["IC_OOS"])
        x["oos_minus_is"] = x["IC_OOS"] - x["IC_IS"]
        x["abs_gap"] = x["oos_minus_is"].abs()
        x["abs_IC_OOS"] = x["IC_OOS"].abs()

        all_rows.append(x)

    if not all_rows:
        return {
            "summary_df": pd.DataFrame(),
            "inconsistent_df": pd.DataFrame(),
        }

    all_df = pd.concat(all_rows, ignore_index=True)

    # ============================================================
    # 1) Resumen exacto para conclusiones
    # ============================================================
    summary_df = (
        all_df.groupby(["regime_name", "horizon"], as_index=False)
        .agg(
            n_total=("indicator", "count"),
            n_same_sign=("same_sign", "sum"),
            pct_same_sign=("same_sign", lambda s: float(s.mean() * 100)),
            mean_IC_IS=("IC_IS", "mean"),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
        )
        .sort_values(["horizon", "regime_name"])
        .reset_index(drop=True)
    )

    # ============================================================
    # 2) Indicadores inconsistentes
    # ============================================================
    inconsistent_df = (
        all_df[~all_df["same_sign"]]
        .sort_values(["horizon", "abs_IC_OOS", "abs_gap"], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    if not inconsistent_df.empty:
        inconsistent_df = (
            inconsistent_df.groupby(["regime_name", "horizon"], group_keys=False)
            .head(top_n_inconsistent)
            .reset_index(drop=True)
        )[
            [
                "regime_name",
                "indicator",
                "horizon",
                "target_col",
                "IC_IS",
                "IC_OOS",
                "oos_minus_is",
                "abs_gap",
                "abs_IC_OOS",
                "same_sign",
            ]
        ]

    return {
        "summary_df": summary_df,
        "inconsistent_df": inconsistent_df,
    }

## **6.2. Aplicarlo a todos los regímenes**

In [97]:
ic_tables = {

    "premarket": ic_table_premarket,
    "opening": ic_table_opening,

}

sign_results = analyze_sign_consistency_across_regimes(ic_tables)

sign_summary_df = sign_results["summary_df"]
sign_inconsistent_df = sign_results["inconsistent_df"]

sign_summary_df

,regime_name,horizon,n_total,n_same_sign,pct_same_sign,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,opening,30,84,84,100.000000,-0.197282,-0.189894,0.228912,0.012024
1,premarket,30,84,77,91.666667,-0.101546,-0.118173,0.134445,0.031376
2,opening,60,42,42,100.000000,-0.210797,-0.190365,0.231750,0.021383
3,premarket,60,42,37,88.095238,-0.100420,-0.108681,0.118005,0.015514


## **6.3. Conclusiones — Consistencia de signo IS vs OOS (targets T2)**

**1. Alta consistencia de signo en general**

* `pct_same_sign` elevado en todos los casos

  * opening: 100%
  * premarket: ~88% – 92%

Esto indica que la relación entre indicadores y target se mantiene estable entre IS y OOS.

---

**2. Régimen opening completamente estable**

* 100% consistencia en ambos horizontes (30 y 60)
* `mean_abs_gap` bajo (~0.01 – 0.02)

Interpretación:

* señal altamente confiable
* sin cambios estructurales entre IS y OOS

---

**3. Régimen premarket con ligera inestabilidad**

* consistencia menor (~88% – 92%)
* `mean_abs_gap` algo más alto en h=30 (~0.03)

Interpretación:

* la señal existe, pero es menos robusta
* algunos indicadores cambian de comportamiento

---

**4. Magnitud de señal coherente con análisis previos**

* `mean_abs_IC_OOS`:

  * opening: ~0.23
  * premarket: ~0.12 – 0.13

Confirma que:

* opening tiene mayor calidad de señal
* premarket es más débil

---

**5. Estabilidad frente al horizonte**

* opening:

  * mantiene 100% consistencia en 30 y 60
* premarket:

  * leve deterioro al cambiar horizonte

Indica que la señal en opening es más estructural.

---

**6. Señal consistente en signo negativo**

* `mean_IC_IS` y `mean_IC_OOS` negativos en todos los casos

Conclusión:

* se mantiene la dinámica de reversión a la media
* no hay inversión de comportamiento fuera de muestra

---

**Conclusión**

* La señal es altamente consistente en signo, especialmente en régimen opening
* No hay evidencia de cambios estructurales importantes entre IS y OOS
* El régimen opening presenta mayor estabilidad y confiabilidad
* El premarket muestra señal válida, pero con mayor variabilidad
* La consistencia de signo refuerza la robustez de los indicadores seleccionados


# **7. Análisis por régimen de mercado**


   
Se evalúan los indicadores en:

* premarket
* opening


Objetivo:

* detectar factores estructurales
* detectar factores contextuales

## **7.1. Código general**

In [102]:
import numpy as np
import pandas as pd

def analyze_regime_dependence(
    ic_tables: dict[str, pd.DataFrame],
    *,
    target_cols: list[str],
    top_n_each_regime: int = 10,
    min_abs_ic_oos: float = 0.05,
    require_same_sign: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Analiza factores estructurales y contextuales por régimen de mercado
    para targets específicos.
    """

    required_cols = [
        "indicator", "target_col",
        "IC_IS", "IC_OOS", "abs_IC_OOS"
    ]

    prepared_rows = []
    top_rows = []

    for regime_name, df in ic_tables.items():
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"En régimen '{regime_name}' faltan columnas: {missing}")

        x = df.copy()
        x = x[x["IC_IS"].notna() & x["IC_OOS"].notna()].copy()
        x = x[x["target_col"].isin(target_cols)].copy()

        if x.empty:
            continue

        x["regime_name"] = regime_name
        x["same_sign"] = np.sign(x["IC_IS"]) == np.sign(x["IC_OOS"])
        x["oos_minus_is"] = x["IC_OOS"] - x["IC_IS"]
        x["abs_gap"] = x["oos_minus_is"].abs()

        x = x[x["abs_IC_OOS"] >= min_abs_ic_oos].copy()

        if require_same_sign:
            x = x[x["same_sign"]].copy()

        if x.empty:
            continue

        prepared_rows.append(x)

        for target in target_cols:
            xt = x[x["target_col"] == target].copy()
            if xt.empty:
                continue

            top_t = (
                xt.sort_values(["abs_IC_OOS", "abs_gap"], ascending=[False, True])
                  .head(top_n_each_regime)
                  .copy()
            )
            top_rows.append(top_t)

    if not prepared_rows:
        return {
            "summary_by_regime_df": pd.DataFrame(),
            "structural_factors_df": pd.DataFrame(),
            "contextual_factors_df": pd.DataFrame(),
            "top_by_regime_df": pd.DataFrame(),
        }

    prepared_df = pd.concat(prepared_rows, ignore_index=True)

    if top_rows:
        top_by_regime_df = pd.concat(top_rows, ignore_index=True)
    else:
        top_by_regime_df = pd.DataFrame()

    summary_by_regime_df = (
        prepared_df.groupby(["regime_name", "target_col"], as_index=False)
        .agg(
            n_indicators=("indicator", "count"),
            mean_IC_IS=("IC_IS", "mean"),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
            pct_same_sign=("same_sign", lambda s: float(s.mean() * 100)),
            best_indicator=("indicator", "first"),
        )
        .sort_values(["target_col", "mean_abs_IC_OOS"], ascending=[True, False])
        .reset_index(drop=True)
    )

    if not top_by_regime_df.empty:
        structural_factors_df = (
            top_by_regime_df.groupby(["target_col", "indicator"], as_index=False)
            .agg(
                n_regimes=("regime_name", "nunique"),
                regimes=("regime_name", lambda s: sorted(set(s))),
                mean_IC_OOS=("IC_OOS", "mean"),
                mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
                mean_abs_gap=("abs_gap", "mean"),
            )
            .sort_values(["target_col", "n_regimes", "mean_abs_IC_OOS"], ascending=[True, False, False])
            .reset_index(drop=True)
        )
    else:
        structural_factors_df = pd.DataFrame()

    if not structural_factors_df.empty:
        contextual_only = structural_factors_df[structural_factors_df["n_regimes"] == 1].copy()

        contextual_factors_df = contextual_only.rename(columns={"regimes": "regime"})[
            ["target_col", "indicator", "regime", "mean_IC_OOS", "mean_abs_IC_OOS", "mean_abs_gap"]
        ].sort_values(["target_col", "mean_abs_IC_OOS"], ascending=[True, False]).reset_index(drop=True)
    else:
        contextual_factors_df = pd.DataFrame()

    return {
        "summary_by_regime_df": summary_by_regime_df,
        "structural_factors_df": structural_factors_df,
        "contextual_factors_df": contextual_factors_df,
        "top_by_regime_df": top_by_regime_df,
    }

In [103]:
target_cols_t2 = [
    "t2_p40_h30",
    "t2_p40_h60",
    "t2_p50_h30",
]

res = analyze_regime_dependence(
    ic_tables=ic_tables_t2,
    target_cols=target_cols_t2,
    top_n_each_regime=10,
    min_abs_ic_oos=0.05,
    require_same_sign=True,
)

## **7.2. Resultados**

In [104]:
print('\n1. Resumen por régimen:\n')
display(res["summary_by_regime_df"])
print('\n2. Factores estructurales:\n')
display(res["structural_factors_df"])
print('\n3. Factores contextuales:\n')
display(res["contextual_factors_df"])
print('\n4. Top usados para el análisis:\n')
display(res["top_by_regime_df"])


1. Resumen por régimen:



,regime_name,target_col,n_indicators,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,best_indicator
0,regime_2,t2_p40_h30,37,-0.230646,-0.215399,0.252086,0.015247,100.0,roc_60
1,regime_1,t2_p40_h30,37,-0.124353,-0.147325,0.164946,0.037068,100.0,roc_60
2,regimes_1_2,t2_p40_h30,36,-0.125832,-0.105059,0.127968,0.020773,100.0,roc_60
3,regime_2,t2_p40_h60,38,-0.236551,-0.214096,0.252453,0.023195,100.0,roc_60
4,regimes_1_2,t2_p40_h60,37,-0.175686,-0.159408,0.177799,0.016277,100.0,roc_60
5,regime_1,t2_p40_h60,32,-0.133279,-0.148763,0.148763,0.015721,100.0,roc_60
6,regime_2,t2_p50_h30,37,-0.229297,-0.223354,0.259966,0.007633,100.0,roc_60
7,regime_1,t2_p50_h30,37,-0.105164,-0.122218,0.138977,0.031874,100.0,roc_60
8,regimes_1_2,t2_p50_h30,36,-0.125288,-0.104441,0.126783,0.020847,100.0,roc_60



2. Factores estructurales:



,target_col,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,t2_p40_h30,roc_60,3,"[regime_1, regime_2, regimes_1_2]",-0.382042,0.382042,0.030421
1,t2_p40_h30,ema_60,3,"[regime_1, regime_2, regimes_1_2]",-0.335108,0.335108,0.031202
2,t2_p40_h30,roc_30,3,"[regime_1, regime_2, regimes_1_2]",-0.274356,0.274356,0.025112
3,t2_p40_h30,ema_30,3,"[regime_1, regime_2, regimes_1_2]",-0.264002,0.264002,0.028452
4,t2_p40_h30,rsi_14,3,"[regime_1, regime_2, regimes_1_2]",-0.246685,0.246685,0.028729
5,t2_p40_h30,bb_60_15,3,"[regime_1, regime_2, regimes_1_2]",-0.236900,0.236900,0.030205
6,t2_p40_h30,bb_60_20,3,"[regime_1, regime_2, regimes_1_2]",-0.236900,0.236900,0.030205
7,t2_p40_h30,stoch_k_30,3,"[regime_1, regime_2, regimes_1_2]",-0.231695,0.231695,0.028391
8,t2_p40_h30,ema_20,2,"[regime_1, regime_2]",-0.258910,0.258910,0.025992
9,t2_p40_h30,roc_20,2,"[regime_2, regimes_1_2]",-0.231955,0.231955,0.013714



3. Factores contextuales:



,target_col,indicator,regime,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,t2_p40_h60,bb_30_15,[regime_1],-0.166185,0.166185,0.021609



4. Top usados para el análisis:



,indicator,target_col,horizon,regime_scope,IC_IS,IC_OOS,oos_minus_is,n_pairs_IS,n_pairs_OOS,abs_IC_OOS,note,regime_name,same_sign,abs_gap
0,roc_60,t2_p40_h30,30,1_2,-0.301845,-0.295472,0.006373,113520,64320,0.295472,,regimes_1_2,True,0.006373
1,ema_60,t2_p40_h30,30,1_2,-0.261883,-0.246606,0.015277,113520,64320,0.246606,,regimes_1_2,True,0.015277
2,roc_30,t2_p40_h30,30,1_2,-0.211742,-0.201806,0.009936,113520,64320,0.201806,,regimes_1_2,True,0.009936
3,ema_30,t2_p40_h30,30,1_2,-0.204020,-0.184025,0.019995,113520,64320,0.184025,,regimes_1_2,True,0.019995
4,bb_60_15,t2_p40_h30,30,1_2,-0.191306,-0.178910,0.012396,113520,64320,0.178910,,regimes_1_2,True,0.012396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,stoch_k_30,t2_p50_h30,30,2,-0.355079,-0.342910,0.012169,56760,32160,0.342910,,regime_2,True,0.012169
86,roc_20,t2_p50_h30,30,2,-0.321413,-0.324106,-0.002693,56760,32160,0.324106,,regime_2,True,0.002693
87,ema_20,t2_p50_h30,30,2,-0.321750,-0.321534,0.000216,56760,32160,0.321534,,regime_2,True,0.000216
88,bb_60_15,t2_p50_h30,30,2,-0.319360,-0.315182,0.004178,56760,32160,0.315182,,regime_2,True,0.004178


## **7.3. Conclusiones — Análisis por régimen de mercado (targets T2)**

**1. El régimen 2 vuelve a destacar como el entorno más fuerte**

En los tres targets, `regime_2` presenta los mayores valores de `mean_abs_IC_OOS` y también gaps bajos entre IS y OOS. Esto indica que no solo hay más señal en opening, sino que además esa señal es estable fuera de muestra.

**2. El régimen 1 tiene señal, pero claramente más débil**

`regime_1` mantiene IC negativos y consistencia de signo del 100%, pero con magnitudes menores que `regime_2`. Esto sugiere que premarket aporta información útil, aunque menos intensa y algo menos robusta que opening.

**3. La mezcla `regimes_1_2` diluye la señal**

La tabla combinada conserva estabilidad, pero su `mean_abs_IC_OOS` queda por debajo de `regime_2` en todos los targets. Esto confirma que agregar ambos regímenes reduce contraste y mezcla dinámicas distintas.

**4. Existen factores estructurales muy claros**

Se repiten en los tres targets y en los tres scopes analizados indicadores como:

* `roc_60`
* `ema_60`
* `roc_30`
* `ema_30`
* `rsi_14`
* `bb_60_15`
* `bb_60_20`

En varios casos también aparece `stoch_k_30`. Esto indica que hay un núcleo de features que no depende de un único target ni de un único régimen, por lo que pueden considerarse factores estructurales.

**5. `roc_60` es el factor estructural dominante**

Aparece como `best_indicator` en todos los targets y regímenes del resumen. Además, lidera la tabla de factores estructurales con los mayores `mean_abs_IC_OOS`. Es, de momento, la feature más consistente y fuerte de todo el análisis.

**6. Los factores contextuales son muy pocos**

Solo aparece un factor contextual claro: `bb_30_15` para `t2_p40_h60` en `regime_1`. Esto sugiere que la mayor parte de la señal no es puramente local o accidental, sino que proviene de un conjunto de indicadores bastante generalizable.

**7. La estructura de señal es muy parecida entre targets**

Los rankings de factores estructurales cambian poco entre `t2_p40_h30`, `t2_p40_h60` y `t2_p50_h30`. Esto refuerza la idea de que la señal capturada por estas features no depende demasiado del target puntual, sino de una dinámica de mercado más estable.

**8. El signo negativo vuelve a ser dominante**

Todos los factores estructurales principales tienen `mean_IC_OOS` negativo. La lectura sigue siendo la misma: las features más útiles están capturando una dinámica predominante de reversión a la media, no de momentum puro.

**Conclusión**

* `opening` es el régimen con mejor calidad de señal y mayor robustez.
* Existe un núcleo claro de factores estructurales compartidos entre targets y regímenes.
* `roc_60`, `ema_60`, `roc_30`, `ema_30` y `rsi_14` son los candidatos más sólidos.
* Los factores contextuales son escasos, lo cual favorece una selección de features más estable.
* La señal observada parece más estructural que circunstancial.


# **8. Ranking de indicadores**


En esta etapa se priorizan los indicadores técnicos utilizando los resultados actualizados del análisis, considerando:

* Magnitud de la señal fuera de muestra (`IC_OOS`)
* Estabilidad entre IS y OOS (`abs_gap`)
* Consistencia de signo
* Presencia en múltiples regímenes (factor estructural)

El objetivo es identificar los indicadores más robustos y generalizables para el modelado.

---

**Criterios de ranking**

Un indicador se considera de alta calidad cuando:

* presenta un `|IC_OOS|` elevado → señal fuerte
* tiene un `abs_gap` bajo → buena estabilidad temporal
* mantiene el mismo signo entre IS y OOS → consistencia direccional
* aparece en múltiples regímenes → robustez estructural

---

**Resultados principales**

**Indicadores mejor rankeados**

Los indicadores que dominan el ranking, de forma consistente en los tres targets y regímenes, son:

* `roc_60`
* `ema_60`
* `roc_30`
* `ema_30`
* `rsi_14`
* `bb_60_15`, `bb_60_20`

Estos presentan:

* `|IC_OOS|` elevado (~0.25 – 0.40, con picos mayores en opening)
* `abs_gap` bajo (~0.01 – 0.03)
* consistencia de signo del 100%
* presencia en todos los regímenes analizados

---

**Dominancia de factores estructurales**

Los indicadores mejor posicionados pertenecen a las siguientes familias:

* retornos: `ROC`
* medias móviles: `EMA`
* osciladores: `RSI`, `Stochastic`
* volatilidad / bandas: `Bollinger Bands`

Esto indica que la señal está dominada por:

* dinámicas de sobre-extensión del precio
* reversión a la media
* suavización de tendencia

---

**Indicadores secundarios**

Otros indicadores relevantes, pero con menor consistencia o menor cobertura:

* `stoch_k_30`
* `ema_20`
* `roc_20`
* `bb_60_25`

Estos:

* aparecen en menos regímenes
* o presentan menor estabilidad
* pueden aportar valor complementario

---

**Factores contextuales**

Se observa una presencia muy limitada de factores puramente contextuales.

Ejemplo:

* `bb_30_15` aparece solo en `premarket` para un target específico

Esto refuerza que la mayor parte de la señal proviene de factores estructurales.

---

**Consistencia entre targets**

El ranking es altamente consistente entre:

* `t2_p40_h30`
* `t2_p40_h60`
* `t2_p50_h30`

Los mismos indicadores dominan en todos los casos, lo que indica:

* baja sensibilidad al horizonte
* baja sensibilidad al percentil del target
* fuerte componente estructural de la señal

---

**Dependencia por régimen**

* `opening` presenta la mayor magnitud de señal
* `premarket` mantiene la misma estructura, pero con menor intensidad
* la combinación de regímenes diluye la señal

Esto sugiere que:

* la calidad del indicador depende del régimen
* pero el ranking relativo se mantiene

---

**Conclusión del ranking**

Se identifica un conjunto reducido de indicadores que concentran la mayor parte de la señal:

* robustos en OOS
* estables entre IS y OOS
* consistentes en signo
* presentes en múltiples regímenes

Estos indicadores constituyen el núcleo del sistema y son los principales candidatos para la etapa de selección de features y modelado.


## **8.1. Códigos de ranking**

In [117]:
# ============================================================
# 1. Ranking global de indicadores
# ============================================================
ranking_df = (
    robustness_top_all
    .groupby("indicator")
    .agg(
        n_regimes=("regime_name", "nunique"),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
        pct_same_sign=("same_sign", "mean"),
    )
    .reset_index()
)

ranking_df["pct_same_sign"] *= 100

ranking_df = ranking_df.sort_values(
    ["mean_abs_IC_OOS", "n_regimes"],
    ascending=[False, False]
).reset_index(drop=True)


# ============================================================
# 2. Ranking por target
# ============================================================
ranking_by_target = (
    robustness_top_all
    .groupby(["target_col", "indicator"])
    .agg(
        n_regimes=("regime_name", "nunique"),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
        pct_same_sign=("same_sign", "mean"),
    )
    .reset_index()
)

ranking_by_target["pct_same_sign"] *= 100

ranking_by_target = ranking_by_target.sort_values(
    ["target_col", "mean_abs_IC_OOS", "n_regimes"],
    ascending=[True, False, False]
).reset_index(drop=True)


# ============================================================
# 3. Top indicadores core
# ============================================================
core_indicators = ranking_df[
    ranking_df["n_regimes"] == 3
].sort_values("mean_abs_IC_OOS", ascending=False)


# ============================================================
# 4. Indicadores secundarios
# ============================================================
secondary_indicators = ranking_df[
    ranking_df["n_regimes"] == 2
].sort_values("mean_abs_IC_OOS", ascending=False)


# ============================================================
# 5. Indicadores contextuales
# ============================================================
contextual_indicators = ranking_df[
    ranking_df["n_regimes"] == 1
].sort_values("mean_abs_IC_OOS", ascending=False)


# ============================================================
# 6. Validación de estabilidad
# ============================================================
stability_top10 = ranking_df[[
    "indicator",
    "mean_abs_IC_OOS",
    "mean_abs_gap",
    "pct_same_sign"
]].sort_values("mean_abs_IC_OOS", ascending=False).head(10)


# ============================================================
# 7. Clasificación por familia
# ============================================================
def classify_family(indicator: str) -> str:
    s = indicator.lower()
    if s.startswith("ema_"):
        return "EMA"
    if s.startswith("roc_"):
        return "ROC"
    if s.startswith("rsi_"):
        return "RSI"
    if s.startswith("bb_"):
        return "BB"
    if s.startswith("stoch_"):
        return "STOCH"
    return "OTHER"


ranking_df["family"] = ranking_df["indicator"].apply(classify_family)

family_summary = (
    ranking_df.groupby("family")
    .agg(
        n_indicators=("indicator", "count"),
        mean_abs_IC_OOS=("mean_abs_IC_OOS", "mean")
    )
    .sort_values("mean_abs_IC_OOS", ascending=False)
)




## **8.2. Resultados**

In [118]:
# ============================================================
# 8. Resultados
# ============================================================
print("\nRanking global de indicadores:\n")
display(ranking_df.head(15))

print("\nRanking por target:\n")
display(ranking_by_target.head(15))

print("\nTop indicadores core:\n")
display(core_indicators)

print("\nIndicadores secundarios:\n")
display(secondary_indicators)

print("\nIndicadores contextuales:\n")
display(contextual_indicators)

print("\nClasificación por familia:\n")
display(family_summary)

print("\nValidación de estabilidad:\n")
display(stability_top10)


Ranking global de indicadores:



,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,family
0,roc_60,3,-0.379127,0.379127,0.025730,100.0,ROC
1,ema_60,3,-0.335433,0.335433,0.024491,100.0,EMA
2,stoch_k_30,2,-0.286405,0.286405,0.024743,100.0,STOCH
3,roc_30,3,-0.286109,0.286109,0.016809,100.0,ROC
4,roc_20,2,-0.277167,0.277167,0.024433,100.0,ROC
5,ema_30,3,-0.274497,0.274497,0.020585,100.0,EMA
6,rsi_14,3,-0.268324,0.268324,0.022263,100.0,RSI
7,ema_20,3,-0.238316,0.238316,0.019277,100.0,EMA
8,bb_60_15,3,-0.228657,0.228657,0.023743,100.0,BB
9,bb_60_20,3,-0.228657,0.228657,0.023743,100.0,BB



Ranking por target:



,target_col,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,t2_p40_h30,roc_60,3,-0.382042,0.382042,0.030421,100.0
1,t2_p40_h30,ema_60,3,-0.335108,0.335108,0.031202,100.0
2,t2_p40_h30,roc_30,3,-0.274356,0.274356,0.025112,100.0
3,t2_p40_h30,ema_30,3,-0.264002,0.264002,0.028452,100.0
4,t2_p40_h30,rsi_14,1,-0.235511,0.235511,0.053547,100.0
5,t2_p40_h30,bb_60_25,1,-0.226132,0.226132,0.062815,100.0
6,t2_p40_h30,bb_60_15,2,-0.202521,0.202521,0.037606,100.0
7,t2_p40_h30,bb_60_20,2,-0.202521,0.202521,0.037606,100.0
8,t2_p40_h60,roc_60,3,-0.387309,0.387309,0.019141,100.0
9,t2_p40_h60,ema_60,3,-0.348713,0.348713,0.016859,100.0



Top indicadores core:



...

,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign



Clasificación por familia:



,n_indicators,mean_abs_IC_OOS
family,,
ROC,3,0.314134
STOCH,1,0.286405
EMA,3,0.282749
RSI,1,0.268324
BB,3,0.226178



Validación de estabilidad:



,indicator,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,roc_60,0.379127,0.025730,100.0
1,ema_60,0.335433,0.024491,100.0
2,stoch_k_30,0.286405,0.024743,100.0
3,roc_30,0.286109,0.016809,100.0
4,roc_20,0.277167,0.024433,100.0
5,ema_30,0.274497,0.020585,100.0
6,rsi_14,0.268324,0.022263,100.0
7,ema_20,0.238316,0.019277,100.0
8,bb_60_15,0.228657,0.023743,100.0
9,bb_60_20,0.228657,0.023743,100.0


## **8.3. Conclusiones del ranking**

**1. Existe un núcleo claro de indicadores dominantes**

Los indicadores con mayor prioridad son:

* `roc_60`
* `ema_60`
* `roc_30`
* `ema_30`

Estos combinan:

* alta magnitud de señal (`|IC_OOS|` ~0.27–0.38)
* baja degradación (`abs_gap` ~0.02)
* presencia en todos los regímenes (`n_regimes = 3`)

Constituyen el núcleo más sólido del sistema.

---

**2. Señal altamente estable**

* `pct_same_sign = 100%` en todos los indicadores relevantes
* `abs_gap` bajo en todos los casos (~0.016 – 0.03)

Indica:

* ausencia de sobreajuste
* excelente generalización IS → OOS

---

**3. Consistencia entre targets**

Los mismos indicadores dominan en:

* `t2_p40_h30`
* `t2_p40_h60`

El ranking cambia muy poco entre targets, lo que implica:

* robustez frente al horizonte
* robustez frente a la definición del target

---

**4. Dominio de la familia ROC y EMA**

Resumen por familias:

* ROC → mayor `mean_abs_IC_OOS` (~0.31)
* EMA → segundo lugar (~0.28)
* STOCH / RSI → nivel intermedio
* BB → menor, pero consistente

Interpretación:

* la señal principal está en:

  * retornos recientes
  * suavización de tendencia

---

**5. Indicadores core bien definidos**

Indicadores estructurales (`n_regimes = 3`):

* ROC: `roc_60`, `roc_30`
* EMA: `ema_60`, `ema_30`, `ema_20`
* RSI: `rsi_14`
* Bollinger: `bb_60_15`, `bb_60_20`

Estos:

* aparecen en todos los regímenes
* mantienen señal OOS
* son candidatos directos para modelado

---

**6. Indicadores secundarios útiles pero no críticos**

* `stoch_k_30`
* `roc_20`
* `bb_60_25`

Características:

* buena señal (`|IC_OOS|` alto)
* menor cobertura entre regímenes (`n_regimes = 2`)

Sirven como complemento, no como base.

---

**7. Ausencia de indicadores puramente contextuales**

* no hay indicadores con `n_regimes = 1`

Esto implica:

* la señal es estructural
* no depende de condiciones específicas de mercado

---

**8. Señal consistentemente negativa**

Todos los `mean_IC_OOS` son negativos.

Interpretación:

* dinámica dominante de reversión a la media
* los indicadores capturan sobre-extensión del precio

---

**9. Estructura jerárquica clara del set de features**

Se puede definir:

* núcleo fuerte → ROC + EMA
* soporte → RSI + Bollinger
* complemento → Stochastic + ROC corto

---

**Conclusión**

* Existe un conjunto compacto y robusto de indicadores que concentran la señal
* La señal es estable, consistente y generalizable
* Los indicadores clave pertenecen a familias bien conocidas (ROC, EMA)
* No hay evidencia de sobreajuste ni dependencia de régimen
* El sistema está listo para pasar a selección final de features y modelado

El siguiente paso natural es reducir redundancia dentro de cada familia (por ejemplo, elegir entre `roc_60`, `roc_30`, etc.).


# **9. Análisis de repetición entre regímenes**
   



   Se identifican indicadores que:

* aparecen consistentemente en varios regímenes

Objetivo:
Detectar factores robustos globales.

## **9.1. Código**

In [120]:
# ============================================================
# 1) FRECUENCIA DE APARICIÓN POR RÉGIMEN
# ============================================================
repetition_df = (
    robustness_top_all
    .groupby("indicator")
    .agg(
        n_regimes=("regime_name", "nunique"),
        regimes=("regime_name", lambda x: sorted(x.unique())),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
    )
    .reset_index()
    .sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])
)

# ============================================================
# 2) Indicadores globales
# ============================================================
global_factors = repetition_df[
    repetition_df["n_regimes"] == 3
].copy()

# ============================================================
# 3) Indicadores semi-globales
# ============================================================
semi_global_factors = repetition_df[
    repetition_df["n_regimes"] == 2
].copy()

# ============================================================
# 4) Indicadores débiles / locales
# ============================================================
local_factors = repetition_df[
    repetition_df["n_regimes"] == 1
].copy()

# ============================================================
# 5) Validación: relación repetición vs señal
# ============================================================
repetition_validation = repetition_df[[
    "indicator",
    "n_regimes",
    "mean_abs_IC_OOS",
    "mean_abs_gap"
]].sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])

# ============================================================
# 9.2. Resultados
# ============================================================
print("\n1. Análisis de repetición entre regímenes")
display(repetition_df)

print("\n2. Indicadores globales")
display(global_factors)

print("\n3. Indicadores semi-globales")
display(semi_global_factors)

print("\n4. Indicadores débiles / locales")
display(local_factors)

print("\n5. Validación: relación repetición vs señal")
display(repetition_validation)


1. Análisis de repetición entre regímenes


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
8,roc_60,3,"[opening, premarket, premarket_opening]",-0.379127,0.379127,0.025730
5,ema_60,3,"[opening, premarket, premarket_opening]",-0.335433,0.335433,0.024491
7,roc_30,3,"[opening, premarket, premarket_opening]",-0.286109,0.286109,0.016809
4,ema_30,3,"[opening, premarket, premarket_opening]",-0.274497,0.274497,0.020585
9,rsi_14,3,"[opening, premarket, premarket_opening]",-0.268324,0.268324,0.022263
3,ema_20,3,"[opening, premarket, premarket_opening]",-0.238316,0.238316,0.019277
0,bb_60_15,3,"[opening, premarket, premarket_opening]",-0.228657,0.228657,0.023743
1,bb_60_20,3,"[opening, premarket, premarket_opening]",-0.228657,0.228657,0.023743
10,stoch_k_30,2,"[opening, premarket]",-0.286405,0.286405,0.024743
6,roc_20,2,"[opening, premarket_opening]",-0.277167,0.277167,0.024433



2. Indicadores globales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
8,roc_60,3,"[opening, premarket, premarket_opening]",-0.379127,0.379127,0.025730
5,ema_60,3,"[opening, premarket, premarket_opening]",-0.335433,0.335433,0.024491
7,roc_30,3,"[opening, premarket, premarket_opening]",-0.286109,0.286109,0.016809
4,ema_30,3,"[opening, premarket, premarket_opening]",-0.274497,0.274497,0.020585
9,rsi_14,3,"[opening, premarket, premarket_opening]",-0.268324,0.268324,0.022263
3,ema_20,3,"[opening, premarket, premarket_opening]",-0.238316,0.238316,0.019277
0,bb_60_15,3,"[opening, premarket, premarket_opening]",-0.228657,0.228657,0.023743
1,bb_60_20,3,"[opening, premarket, premarket_opening]",-0.228657,0.228657,0.023743



3. Indicadores semi-globales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
10,stoch_k_30,2,"[opening, premarket]",-0.286405,0.286405,0.024743
6,roc_20,2,"[opening, premarket_opening]",-0.277167,0.277167,0.024433
2,bb_60_25,2,"[premarket, premarket_opening]",-0.221222,0.221222,0.028411



4. Indicadores débiles / locales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap



5. Validación: relación repetición vs señal


,indicator,n_regimes,mean_abs_IC_OOS,mean_abs_gap
8,roc_60,3,0.379127,0.025730
5,ema_60,3,0.335433,0.024491
7,roc_30,3,0.286109,0.016809
4,ema_30,3,0.274497,0.020585
9,rsi_14,3,0.268324,0.022263
3,ema_20,3,0.238316,0.019277
0,bb_60_15,3,0.228657,0.023743
1,bb_60_20,3,0.228657,0.023743
10,stoch_k_30,2,0.286405,0.024743
6,roc_20,2,0.277167,0.024433


## **9.3. Análisis de repetición entre regímenes**


**1. Predominio claro de factores estructurales**

* La mayoría de los indicadores relevantes tienen `n_regimes = 3`
* No existen indicadores con `n_regimes = 1`

Esto indica que:

* la señal es global
* no depende de condiciones específicas de mercado

---

**2. Núcleo robusto bien definido**

Indicadores globales:

* `roc_60`
* `ema_60`
* `roc_30`
* `ema_30`
* `rsi_14`
* `ema_20`
* `bb_60_15`, `bb_60_20`

Estos cumplen simultáneamente:

* alta señal (`|IC_OOS|` alto)
* estabilidad (`abs_gap` bajo)
* presencia en todos los regímenes

---

**3. Liderazgo claro de ROC y EMA**

* `roc_60` y `ema_60` dominan el ranking
* `roc_30` y `ema_30` refuerzan esa estructura

Interpretación:

* la información relevante está en:

  * retornos recientes
  * tendencia suavizada

---

**4. Indicadores semi-globales con buen desempeño**

* `stoch_k_30`
* `roc_20`
* `bb_60_25`

Observación importante:

* algunos tienen magnitudes comparables a los core (`~0.28`)
* pero menor cobertura de regímenes

Conclusión:

* son útiles como complemento
* no forman parte del núcleo principal

---

**5. Relación clara entre repetición y robustez**

En la validación:

* los indicadores con `n_regimes = 3` dominan el ranking
* mantienen alto `|IC_OOS|` y bajo `abs_gap`

Esto confirma:

* mayor repetición → mayor robustez

---

**6. Señal consistentemente negativa**

* todos los `mean_IC_OOS` son negativos

Interpretación:

* dinámica dominante de reversión a la media
* consistente en todos los regímenes

---

**7. Ausencia de factores contextuales**

* no hay indicadores con `n_regimes = 1`

Esto implica:

* no hay dependencia de condiciones particulares
* el sistema se apoya en patrones estables

---

**Conclusión**

* La señal es principalmente estructural y consistente entre regímenes
* Existe un núcleo reducido de indicadores que concentra la mayor parte del poder predictivo
* ROC y EMA lideran claramente el comportamiento del sistema
* Los indicadores secundarios pueden aportar valor, pero no son críticos
* La robustez observada refuerza la validez de estos factores para el modelado

El siguiente paso lógico es reducir redundancia dentro del núcleo de indicadores.


# **10. Identificación de indicadores específicos por régimen**

Este análisis busca identificar indicadores que no solo presenten señal robusta, sino que además sean característicos de un régimen particular. A diferencia de los factores estructurales, que se repiten en varios contextos de mercado, aquí el objetivo es detectar señales contextuales: indicadores que funcionan bien en un régimen y no aparecen entre los más relevantes en los demás.

La lógica es simple: se toman los indicadores mejor rankeados de cada régimen y se buscan aquellos que aparecen únicamente en el top de un régimen, sin repetirse en los otros. Si un indicador cumple esta condición, se considera específico de ese contexto.

## **10.1. Código**

In [ ]:
import pandas as pd

def find_regime_specific_indicators(
    robustness_top_all: pd.DataFrame,
    *,
    top_n_per_regime: int = 10,
) -> pd.DataFrame:
    """
    Identifica indicadores específicos por régimen:
    aparecen en el top de un régimen y no aparecen
    en el top de ningún otro régimen.

    Parámetros
    ----------
    robustness_top_all : pd.DataFrame
        DataFrame combinado con los top robustos de todos los regímenes.
        Debe contener al menos:
        - regime_name
        - indicator
        - IC_OOS
        - abs_IC_OOS
        - abs_gap

    top_n_per_regime : int
        Cantidad de indicadores top a considerar por cada régimen.

    Devuelve
    --------
    pd.DataFrame
        Tabla con indicadores específicos por régimen.
        Puede devolver DataFrame vacío, lo cual significa que
        no hubo indicadores exclusivos.
    """
    required_cols = {"regime_name", "indicator", "IC_OOS", "abs_IC_OOS", "abs_gap"}
    missing = required_cols - set(robustness_top_all.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")

    # 1) Tomar top N por régimen
    top_by_regime = (
        robustness_top_all
        .sort_values(["regime_name", "abs_IC_OOS"], ascending=[True, False])
        .groupby("regime_name", group_keys=False)
        .head(top_n_per_regime)
        .copy()
    )

    # 2) Construir sets de indicadores por régimen
    sets_by_regime = {
        regime: set(g["indicator"].tolist())
        for regime, g in top_by_regime.groupby("regime_name")
    }

    # 3) Buscar exclusivos
    rows = []

    for regime, current_set in sets_by_regime.items():
        other_sets = [s for r, s in sets_by_regime.items() if r != regime]
        others_union = set().union(*other_sets) if other_sets else set()

        specific_indicators = sorted(current_set - others_union)

        if not specific_indicators:
            continue

        df_reg = top_by_regime[top_by_regime["regime_name"] == regime].copy()

        df_reg = df_reg[df_reg["indicator"].isin(specific_indicators)].copy()

        for _, row in df_reg.iterrows():
            rows.append({
                "regime": regime,
                "indicator": row["indicator"],
                "IC_OOS": row["IC_OOS"],
                "abs_IC_OOS": row["abs_IC_OOS"],
                "abs_gap": row["abs_gap"],
            })

    # 4) Armar salida robusta
    if len(rows) == 0:
        return pd.DataFrame(
            columns=["regime", "indicator", "IC_OOS", "abs_IC_OOS", "abs_gap"]
        )

    out = pd.DataFrame(rows).sort_values(
        ["regime", "abs_IC_OOS"],
        ascending=[True, False]
    ).reset_index(drop=True)

    return out

In [ ]:
specific_by_regime = find_regime_specific_indicators(
    robustness_top_all,
    top_n_per_regime=10,
)



##**10.2. Resultados**

In [ ]:
specific_by_regime

,regime,indicator,IC_OOS,abs_IC_OOS,abs_gap
0,regular,rsi_14,-0.198804,0.198804,0.010357


## **10.3. Observaciones**

El resultado obtenido muestra un único indicador específico por régimen:

- rsi_14 en el régimen regular

Este indicador presenta:

- IC_OOS = -0.198804  
- |IC_OOS| = 0.198804  
- abs_gap = 0.010357  

**Interpretación**

Este resultado indica que rsi_14 tiene un comportamiento particularmente relevante en el régimen regular, ya que aparece entre los mejores indicadores de ese contexto, pero no logra posicionarse de la misma manera en los tops de los demás regímenes.

Además, su magnitud de señal es relativamente alta y su gap es bajo, lo que sugiere que no se trata de una señal espuria, sino de una relación estable y específica de ese entorno de mercado.

El signo negativo de IC_OOS refuerza la lectura general observada en el resto del análisis: valores altos del indicador se asocian con movimientos negativos significativos del target T2, lo que sigue siendo consistente con una dinámica de reversión a la media.

**Lectura metodológica**

Que solo aparezca un indicador exclusivo no es un resultado débil. Por el contrario, tiene una lectura importante:

- la mayor parte de la señal del sistema es estructural y compartida entre regímenes,
- existen pocos indicadores realmente exclusivos,
- las señales específicas por régimen son más escasas que las señales globales.

Esto es coherente con los resultados anteriores, donde ya se observó que un núcleo reducido de indicadores domina el ranking en casi todos los contextos de mercado.

**Implicación práctica**

Desde el punto de vista del modelado, este hallazgo sugiere que:

- el feature set principal debe construirse a partir de factores estructurales,
- los factores específicos por régimen pueden incorporarse como complemento,
- no conviene sobrecargar el modelo con demasiados indicadores contextuales si su evidencia es limitada.

En este caso, rsi_14 en régimen regular es un buen candidato para ser tratado como una feature contextual adicional, pero no reemplaza al núcleo principal de indicadores robustos.

**Conclusión**

El análisis de indicadores específicos por régimen muestra que la señal contextual existe, pero es mucho más limitada que la señal estructural. El único factor claramente exclusivo identificado es rsi_14 en el régimen regular, con buena magnitud y estabilidad fuera de muestra.

Esto refuerza la idea de que el sistema está dominado por factores globales y robustos, mientras que los indicadores específicos por régimen deben utilizarse de manera complementaria y selectiva.

# **11. Análisis de redundancia (correlación entre indicadores)**

En esta etapa se analiza la relación entre los indicadores técnicos con el objetivo de detectar redundancia dentro del conjunto de features. Dado que muchos indicadores pertenecen a la misma familia (por ejemplo, EMA, ROC o Bollinger Bands), es esperable que exista una alta correlación entre ellos.

El análisis se realiza sobre el dataset **`mnq_intraday_t2_ti`**, el cual contiene los indicadores técnicos previamente calculados y utilizados en las etapas anteriores de evaluación (IC, robustez y ranking).

El objetivo principal es:

* identificar indicadores que contienen información duplicada
* reducir la dimensionalidad del problema
* evitar problemas de multicolinealidad en los modelos

En modelos lineales como Logistic Regression, la multicolinealidad puede generar:

* inestabilidad en los coeficientes
* dificultad para interpretar la importancia de las variables
* degradación en la capacidad de generalización

En modelos más complejos, aunque el impacto es menor, la redundancia sigue siendo perjudicial porque introduce ruido y aumenta innecesariamente la complejidad del modelo.

---

**Enfoque metodológico**

El análisis se realiza mediante:

* cálculo de la matriz de correlación entre indicadores (correlación de Pearson)
* identificación de pares de features altamente correlacionadas
* construcción de grupos de redundancia (features conectadas entre sí)

Se define un **umbral de correlación absoluta** para detectar redundancia:

* |correlación| ≥ 0.90 → indicadores altamente redundantes

Interpretación:

* correlación alta (≈ 0.90 – 1.00 o -0.90 – -1.00) → información prácticamente duplicada
* correlación media → solapamiento parcial
* correlación baja → información complementaria

Este análisis no busca eliminar indicadores de forma directa, sino agruparlos para seleccionar representantes dentro de cada conjunto redundante.

---

**Lectura esperada en este proyecto**

Dado lo observado en las etapas anteriores, es esperable encontrar:

* alta correlación dentro de familias:

  * EMA (`ema_20`, `ema_30`, `ema_60`)
  * ROC (`roc_20`, `roc_30`, `roc_60`)
  * Bollinger Bands (`bb_60_15`, `bb_60_20`, `bb_60_25`)
* correlación moderada entre indicadores de momentum y osciladores (RSI, Stochastic)
* menor correlación de indicadores que capturan dinámicas distintas (por ejemplo, volatilidad o volumen, si están presentes)

Esto implica que muchos de los indicadores previamente identificados como relevantes no son independientes, sino diferentes formas de medir la misma dinámica de mercado.

---

**Estrategia de reducción**

A partir de este análisis, la estrategia consiste en:

* identificar grupos de indicadores altamente correlacionados
* seleccionar un único representante por grupo

El criterio de selección del representante se basa en:

* mayor `|IC_OOS|` (mayor capacidad predictiva fuera de muestra)
* menor `abs_gap` (mayor estabilidad IS vs OOS)
* mayor presencia en regímenes (`n_regimes`)

Por ejemplo:

* de la familia EMA → seleccionar uno (ej: `ema_60`)
* de la familia ROC → seleccionar uno (ej: `roc_60`)
* de la familia BB → seleccionar uno (ej: `bb_60_20`)

De esta forma, se conserva la señal relevante evitando redundancia.

---

**Conclusión**

El análisis de correlación es un paso clave para transformar un conjunto amplio de indicadores en un feature set compacto y eficiente. Permite mantener la información relevante mientras se eliminan variables redundantes, mejorando la estabilidad, interpretabilidad y desempeño del modelo.

El siguiente paso consiste en construir la matriz de correlación, identificar grupos de redundancia y seleccionar los representantes finales de cada grupo.



## **11.1. Código**

### **1) Seleccionar indicadores relevantes (los que ya validaste)**

In [121]:
import numpy as np
import pandas as pd

def analyze_all_feature_redundancy(
    df: pd.DataFrame,
    *,
    indicator_columns: list[str] | None = None,
    corr_threshold: float = 0.90,
    ranking_df: pd.DataFrame | None = None,
    print_report: bool = True,
) -> dict:
    """
    Analiza redundancia entre TODOS los indicadores técnicos.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset con indicadores técnicos.
    indicator_columns : list[str] | None
        Lista explícita de indicadores a evaluar.
        Si es None, se infiere automáticamente.
    corr_threshold : float
        Umbral de correlación absoluta para considerar redundancia.
    ranking_df : pd.DataFrame | None
        Opcional. Si se pasa, se usa mean_abs_IC_OOS para elegir
        el mejor representante dentro de cada grupo redundante.
        Debe tener columnas: indicator, mean_abs_IC_OOS
    print_report : bool
        Si True, imprime resúmenes.

    Devuelve
    --------
    dict con:
    - feature_list
    - family_summary_df
    - corr_matrix
    - high_corr_pairs_df
    - redundancy_groups_df
    - representatives_df
    """

    # ============================================================
    # 1) Definir TODOS los indicadores a evaluar
    # ============================================================
    if indicator_columns is None:
        exclude_cols = {
            "date", "minute_of_day",
            "open", "high", "low", "close", "volume",
            "delta_60", "delta_90",
            "ret_60", "ret_90",
            "split_fe",
            "is_overnight", "is_premarket", "is_opening", "is_regular", "is_closing",
            "is_mon", "is_tue", "is_wed", "is_thu", "is_fri",
        }

        indicator_columns = [
            c for c in df.columns
            if c not in exclude_cols
        ]

    # conservar solo columnas existentes y numéricas
    indicator_columns = [
        c for c in indicator_columns
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
    ]

    if len(indicator_columns) == 0:
        raise ValueError("No se encontraron indicadores técnicos válidos para analizar.")

    # ============================================================
    # 2) Clasificación por familia
    # ============================================================
    def classify_family(indicator: str) -> str:
        s = indicator.lower()
        if s.startswith("ema_"):
            return "EMA"
        if s.startswith("roc_"):
            return "ROC"
        if s.startswith("rsi_"):
            return "RSI"
        if s.startswith("bb_"):
            return "BB"
        if s.startswith("stoch_"):
            return "STOCH"
        if s.startswith("atr_"):
            return "ATR"
        if s.startswith("mom_"):
            return "MOM"
        if "volume" in s:
            return "VOLUME"
        if "macd" in s:
            return "MACD"
        return "OTHER"

    feature_df = pd.DataFrame({
        "indicator": indicator_columns,
        "family": [classify_family(c) for c in indicator_columns],
    })

    family_summary_df = (
        feature_df.groupby("family", as_index=False)
        .agg(
            n_indicators=("indicator", "count"),
            indicators=("indicator", lambda s: sorted(s.tolist()))
        )
        .sort_values(["n_indicators", "family"], ascending=[False, True])
        .reset_index(drop=True)
    )

    # ============================================================
    # 3) Matriz de correlación
    # ============================================================
    corr_matrix = df[indicator_columns].corr()

    # ============================================================
    # 4) Pares altamente correlacionados
    # ============================================================
    high_corr_rows = []

    for i in range(len(indicator_columns)):
        for j in range(i + 1, len(indicator_columns)):
            f1 = indicator_columns[i]
            f2 = indicator_columns[j]
            corr = corr_matrix.loc[f1, f2]

            if pd.notna(corr) and abs(corr) >= corr_threshold:
                high_corr_rows.append({
                    "feature_1": f1,
                    "family_1": classify_family(f1),
                    "feature_2": f2,
                    "family_2": classify_family(f2),
                    "correlation": corr,
                    "abs_correlation": abs(corr),
                })

    high_corr_pairs_df = pd.DataFrame(high_corr_rows)

    if not high_corr_pairs_df.empty:
        high_corr_pairs_df = high_corr_pairs_df.sort_values(
            ["abs_correlation", "feature_1", "feature_2"],
            ascending=[False, True, True]
        ).reset_index(drop=True)

    # ============================================================
    # 5) Grupos de redundancia (componentes conectadas)
    # ============================================================
    adjacency = {f: set() for f in indicator_columns}

    if not high_corr_pairs_df.empty:
        for _, row in high_corr_pairs_df.iterrows():
            f1 = row["feature_1"]
            f2 = row["feature_2"]
            adjacency[f1].add(f2)
            adjacency[f2].add(f1)

    visited = set()
    groups = []

    for f in indicator_columns:
        if f in visited:
            continue

        stack = [f]
        component = []

        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            component.append(node)

            for neigh in adjacency[node]:
                if neigh not in visited:
                    stack.append(neigh)

        groups.append(sorted(component))

    redundancy_rows = []
    for g_id, group in enumerate(groups, start=1):
        redundancy_rows.append({
            "group_id": g_id,
            "n_features": len(group),
            "features": group,
            "families": sorted(set(classify_family(x) for x in group)),
        })

    redundancy_groups_df = (
        pd.DataFrame(redundancy_rows)
        .sort_values(["n_features", "group_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

    # ============================================================
    # 6) Elegir representante por grupo
    # ============================================================
    ranking_map = {}
    if ranking_df is not None:
        if {"indicator", "mean_abs_IC_OOS"}.issubset(ranking_df.columns):
            ranking_map = dict(
                zip(ranking_df["indicator"], ranking_df["mean_abs_IC_OOS"])
            )

    rep_rows = []
    for _, row in redundancy_groups_df.iterrows():
        group = row["features"]

        if len(group) == 1:
            representative = group[0]
            reason = "único en el grupo"
        else:
            if ranking_map:
                representative = max(group, key=lambda x: ranking_map.get(x, -np.inf))
                reason = "mayor mean_abs_IC_OOS dentro del grupo"
            else:
                representative = group[0]
                reason = "primer indicador del grupo (sin ranking externo)"

        rep_rows.append({
            "group_id": row["group_id"],
            "representative": representative,
            "n_features_in_group": row["n_features"],
            "group_features": group,
            "reason": reason,
            "representative_family": classify_family(representative),
            "representative_mean_abs_IC_OOS": ranking_map.get(representative, np.nan),
        })

    representatives_df = pd.DataFrame(rep_rows)

    # ============================================================
    # 7) Reporte
    # ============================================================
    if print_report:
        print("=" * 100)
        print("ANÁLISIS DE REDUNDANCIA ENTRE TODOS LOS INDICADORES TÉCNICOS")
        print("=" * 100)
        print(f"Cantidad total de features evaluadas: {len(indicator_columns)}")
        print(f"Umbral de correlación absoluta: {corr_threshold:.2f}")
        print(f"Pares altamente correlacionados encontrados: {len(high_corr_pairs_df)}")
        print(f"Cantidad de grupos de redundancia: {len(redundancy_groups_df)}")
        print()

        print("-" * 100)
        print("LISTADO TOTAL DE FEATURES EVALUADAS")
        print("-" * 100)
        print(indicator_columns)
        print()

        print("-" * 100)
        print("RESUMEN POR FAMILIA")
        print("-" * 100)
        print(family_summary_df[["family", "n_indicators"]].to_string(index=False))
        print()

        if not high_corr_pairs_df.empty:
            print("-" * 100)
            print("TOP PARES ALTAMENTE CORRELACIONADOS")
            print("-" * 100)
            print(
                high_corr_pairs_df.head(20)[
                    ["feature_1", "feature_2", "correlation", "abs_correlation"]
                ].to_string(index=False)
            )
            print()
        else:
            print("[OK] No se encontraron pares con correlación superior al umbral.")
            print()

        print("-" * 100)
        print("GRUPOS DE REDUNDANCIA")
        print("-" * 100)
        print(
            redundancy_groups_df[["group_id", "n_features", "families", "features"]]
            .head(20)
            .to_string(index=False)
        )
        print()

        print("-" * 100)
        print("REPRESENTANTES PROPUESTOS POR GRUPO")
        print("-" * 100)
        print(
            representatives_df[
                ["group_id", "representative", "representative_family", "n_features_in_group", "reason"]
            ].to_string(index=False)
        )

    return {
        "feature_list": indicator_columns,
        "family_summary_df": family_summary_df,
        "corr_matrix": corr_matrix,
        "high_corr_pairs_df": high_corr_pairs_df,
        "redundancy_groups_df": redundancy_groups_df,
        "representatives_df": representatives_df,
    }

### **2) Sin ranking externo**

In [122]:
redundancy_results = analyze_all_feature_redundancy(
    mnq_intraday_t2_ti,
    indicator_columns=ti_cols,
    corr_threshold=0.90,
    print_report=True,
)

ANÁLISIS DE REDUNDANCIA ENTRE TODOS LOS INDICADORES TÉCNICOS
Cantidad total de features evaluadas: 42
Umbral de correlación absoluta: 0.90
Pares altamente correlacionados encontrados: 100
Cantidad de grupos de redundancia: 9

----------------------------------------------------------------------------------------------------
LISTADO TOTAL DE FEATURES EVALUADAS
----------------------------------------------------------------------------------------------------
['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3', 'mom_10', 'mom_5', 'mom_3', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90', 'macd', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15', 'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20', 'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25', 'atr_norm_5', 'atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60']

-----------------

**1. Alta redundancia en el dataset**

* 42 features se agrupan en solo 9 grupos
* Se detectan 100 pares altamente correlacionados

Existe una gran cantidad de información duplicada dentro del conjunto de variables.

---

**2. Bollinger, RSI y Stochastic están fuertemente correlacionados**

* Un único grupo concentra 19 indicadores
* La correlación entre ellos es prácticamente perfecta

Estos indicadores están capturando esencialmente la misma dinámica de mercado.

---

**3. EMA, ROC y MOM también presentan alta correlación**

* Un mismo grupo combina indicadores de estas tres familias

Esto indica que no son independientes, sino distintas transformaciones de información similar.

---

**4. Existen correlaciones perfectas**

Ejemplos:

* `mom_5` y `roc_5`
* múltiples configuraciones de Bollinger Bands

Esto implica la presencia de variables completamente redundantes.

---

**5. ATR y Volume forman grupos independientes**

* ATR forma un grupo compacto
* Volume forma otro grupo separado

Cada familia aporta información distinta, pero internamente contiene redundancia.

---

**6. Pocos indicadores realmente independientes**

Indicadores que aparecen como grupos individuales:

* `roc_30`
* `roc_60`
* `macd`
* `mom_3`

Estos son los únicos que aportan información no redundante.

---

**7. Selección sin ranking no es confiable**

Los representantes elegidos sin ranking externo dependen del orden de aparición, por ejemplo:

* `bb_15_15`
* `ema_15`

No necesariamente son los mejores indicadores disponibles.

---

**Conclusión**

* El dataset presenta una alta redundancia entre variables
* Muchas features están midiendo exactamente la misma información
* Es necesario utilizar el ranking para seleccionar representantes adecuados
* Sin una reducción de dimensionalidad, el modelo incorporará ruido innecesario y complejidad adicional


### **3) Usando ranking global para elegir mejor representante por grupo**

In [124]:
redundancy_results = analyze_all_feature_redundancy(
    mnq_intraday_t2_ti,
    indicator_columns=ti_cols,
    corr_threshold=0.90,
    ranking_df=ranking_df,
    print_report=True,
)

ANÁLISIS DE REDUNDANCIA ENTRE TODOS LOS INDICADORES TÉCNICOS
Cantidad total de features evaluadas: 42
Umbral de correlación absoluta: 0.90
Pares altamente correlacionados encontrados: 100
Cantidad de grupos de redundancia: 9

----------------------------------------------------------------------------------------------------
LISTADO TOTAL DE FEATURES EVALUADAS
----------------------------------------------------------------------------------------------------
['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3', 'mom_10', 'mom_5', 'mom_3', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90', 'macd', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15', 'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20', 'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25', 'atr_norm_5', 'atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60']

-----------------

**1. La estructura de redundancia no cambia**

* Siguen siendo 42 features organizadas en 9 grupos
* Se mantienen 100 pares altamente correlacionados

El dataset continúa siendo altamente redundante.

---

**2. La diferencia clave está en los representantes**

Ahora los representantes se seleccionan en base a calidad (`mean_abs_IC_OOS`), no por orden de aparición.

Ejemplos:

* antes: `bb_15_15` → ahora: `stoch_k_30`
* antes: `ema_15` → ahora: `ema_60`

Los indicadores seleccionados ahora tienen sentido predictivo.

---

**3. Mejora clara en el grupo más grande**

Grupo 1 (19 features):

* antes: selección arbitraria
* ahora: `stoch_k_30`

Esto indica que:

* dentro de BB, RSI y STOCH, el indicador más fuerte es STOCH
* el resto resulta redundante frente a este

---

**4. EMA y ROC correctamente representados**

Grupo 2:

* se selecciona `ema_60`

Adicionalmente:

* `roc_30` y `roc_60` aparecen como grupos independientes

Esto es consistente con el ranking previo.

---

**5. MOM y ROC siguen parcialmente duplicados**

Grupo 3:

* `mom_5` y `roc_5` tienen correlación perfecta
* se selecciona `mom_5`

Ambos indicadores son equivalentes en información.

---

**6. ATR y Volume se mantienen como familias independientes**

* cada familia forma su propio grupo
* se selecciona un único representante por familia

Aportan información distinta, aunque con redundancia interna.

---

**7. Existen indicadores realmente independientes**

Se mantienen como grupos individuales:

* `roc_30`
* `roc_60`
* `macd`
* `mom_3`

Estos aportan información no redundante.

---

**8. Reducción significativa del feature set**

Se pasa de:

* 42 features

a aproximadamente:

* 9 representantes

Se logra una reducción de dimensionalidad importante.

---

**9. Coherencia con el análisis previo**

Los indicadores seleccionados son consistentes con:

* el ranking por IC
* el análisis por régimen
* la robustez IS vs OOS

El pipeline completo mantiene coherencia interna.

---

**Conclusión**

* La selección de features ahora es consistente con la señal observada
* Se elimina redundancia manteniendo la información relevante
* El conjunto resultante es compacto, interpretable y robusto
* Se obtiene una base adecuada para la etapa de modelado


## **11.2. Resultados**

In [125]:
print('\nRepresentantes sugeridos\n')
display(redundancy_results["representatives_df"])


Representantes sugeridos



,group_id,representative,n_features_in_group,group_features,reason,representative_family,representative_mean_abs_IC_OOS
0,1,stoch_k_30,19,"[bb_15_15, bb_15_20, bb_15_25, bb_20_15, bb_20...",mayor mean_abs_IC_OOS dentro del grupo,STOCH,0.286405
1,2,ema_60,7,"[ema_15, ema_20, ema_30, ema_60, mom_10, roc_1...",mayor mean_abs_IC_OOS dentro del grupo,EMA,0.335433
2,5,volume_ratio_15,5,"[volume_ratio_15, volume_ratio_20, volume_rati...",mayor mean_abs_IC_OOS dentro del grupo,VOLUME,NaN
3,7,atr_norm_10,5,"[atr_norm_10, atr_norm_14, atr_norm_20, atr_no...",mayor mean_abs_IC_OOS dentro del grupo,ATR,NaN
4,3,mom_5,2,"[mom_5, roc_5]",mayor mean_abs_IC_OOS dentro del grupo,MOM,NaN
5,4,mom_3,1,[mom_3],único en el grupo,MOM,NaN
6,6,macd,1,[macd],único en el grupo,MACD,NaN
7,8,roc_30,1,[roc_30],único en el grupo,ROC,0.286109
8,9,roc_60,1,[roc_60],único en el grupo,ROC,0.379127


['stoch_k_30',
 'roc_20',
 'volume_ratio_15',
 'atr_norm_10',
 'mom_5',
 'mom_3',
 'macd',
 'roc_30',
 'roc_60']

* Se confirma **redundancia extrema**: 42 features → solo **9 grupos efectivos**
* El grupo más grande (BB + RSI + STOCH) mezcla familias distintas → están capturando **la misma estructura de mercado (reversión/momentum corto)**
* Hay correlación perfecta (1.0) entre muchos indicadores → varios son **duplicados exactos con distinta parametrización**
* MOM y ROC están prácticamente duplicados (ej: mom_5 = roc_5) → **uno sobra completamente**
* ATR forma un grupo propio → aporta **información distinta (volatilidad)**
* Volume_ratio también es independiente → posible señal adicional no redundante

Sobre los representantes (clave):

* `stoch_k_30` resume todo BB + RSI + STOCH → fuerte candidato
* `roc_20` representa EMA + ROC + MOM (grupo tendencia)
* `roc_30` y `roc_60` quedaron independientes → aportan señal adicional (no redundantes)
* `atr_norm_10` → captura volatilidad
* `volume_ratio_15` → captura volumen

Conclusión directa:

* Pasaste de 42 features a ~**8–9 realmente útiles**
* La selección ya no es arbitraria: está **alineada con señal (IC)**
* El dataset original estaba **fuertemente sobreparametrizado**

## **11.3. Análisis de IC de features resultantes contra targets T2**

In [ ]:
def compute_ic_by_day(df, feature, target):
    ic_list = []

    for date, g in df.groupby("date"):
        if g[feature].notna().sum() > 10:
            ic = g[feature].corr(g[target])
            if pd.notna(ic):
                ic_list.append(ic)

    return np.mean(ic_list)

In [ ]:
targets_cols

['t2_dir_thr_30', 't2_dir_thr_60', 't2_dir_thr_90', 't2_dir_thr_120']

In [ ]:
results = []

features_to_check = redundancy_results["representatives_df"]['representative'].to_list()

for f in features_to_check:
    for target in targets_cols:
        ic = compute_ic_by_day(
            mnq_intraday_t2_ti,
            f,
            target
        )

        results.append({
            "feature": f,
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic)
        })

pd.DataFrame(results).sort_values("abs_IC", ascending=False)

,feature,target,IC,abs_IC
35,roc_60,t2_dir_thr_120,-0.160679,0.160679
34,roc_60,t2_dir_thr_90,-0.127682,0.127682
31,roc_30,t2_dir_thr_120,-0.117993,0.117993
33,roc_60,t2_dir_thr_60,-0.102934,0.102934
7,roc_20,t2_dir_thr_120,-0.100025,0.100025
30,roc_30,t2_dir_thr_90,-0.096284,0.096284
3,stoch_k_30,t2_dir_thr_120,-0.092760,0.092760
6,roc_20,t2_dir_thr_90,-0.081087,0.081087
14,atr_norm_10,t2_dir_thr_90,0.079695,0.079695
32,roc_60,t2_dir_thr_30,-0.077921,0.077921


## **11.4. Conclusiones del análisis de redundancia y evaluación de IC por target**

El análisis de redundancia permitió reducir un conjunto inicial amplio de indicadores técnicos a un subconjunto mucho más compacto y representativo. A partir de 42 features originales, se identificaron 9 grupos de alta correlación, lo que evidencia una fuerte duplicación estructural dentro del dataset.

Se observó que muchas variables pertenecientes a distintas familias (por ejemplo, Bollinger Bands, RSI y Stochastic) estaban altamente correlacionadas entre sí, indicando que capturan dinámicas similares del mercado, principalmente asociadas a reversión a la media y momentum de corto plazo. Asimismo, indicadores como MOM y ROC resultaron prácticamente equivalentes en varios casos, confirmando redundancia directa.

El uso de un ranking externo basado en IC permitió seleccionar representantes óptimos dentro de cada grupo, eliminando la arbitrariedad en la selección. Esto dio lugar a un conjunto reducido de indicadores con respaldo empírico en términos de señal fuera de muestra y robustez.

Posteriormente, se evaluó la relación entre estos indicadores seleccionados y los distintos targets T2 mediante el Information Coefficient. Este análisis permitió comparar directamente la calidad predictiva de cada horizonte.

Los resultados muestran un patrón claro: la magnitud del IC aumenta de forma consistente con el horizonte de predicción. En particular:

- t2_dir_thr_120 presenta los valores más altos de IC (hasta ~0.16), lo que indica la mayor capacidad predictiva  
- t2_dir_thr_90 muestra también señal sólida (~0.12 – 0.13)  
- t2_dir_thr_60 presenta señal moderada (~0.10)  
- t2_dir_thr_30 tiene señal débil (~0.07)  

Este comportamiento es coherente con la naturaleza del mercado intradía, donde horizontes más cortos están dominados por ruido, mientras que horizontes más largos permiten que las dinámicas de mercado se desarrollen con mayor claridad.

**En cuanto a los indicadores:**

- ROC (especialmente roc_60) emerge como el factor dominante, con la mayor capacidad predictiva en todos los horizontes  
- Stochastic (stoch_k_30) muestra señal consistente, aunque inferior a ROC  
- ATR (atr_norm_*) aporta información complementaria, asociada a la magnitud del movimiento más que a la dirección  
- Volume_ratio presenta una señal prácticamente nula, por lo que no aporta valor al modelo  

Estos resultados confirman que la señal del sistema está principalmente concentrada en factores de momentum, con contribuciones secundarias de osciladores y volatilidad.

**Conclusión general**

El pipeline de selección de features ha permitido:

- eliminar redundancia estructural significativa  
- identificar un conjunto reducido de indicadores realmente informativos  
- validar que la señal predictiva depende fuertemente del horizonte  

Se concluye que:

- no todos los targets son igualmente predecibles  
- los horizontes largos (90 y especialmente 120 minutos) concentran la mayor señal  
- la información relevante del sistema puede capturarse con un número reducido de features  

En consecuencia, el modelado debe centrarse en:

- targets de horizonte largo (principalmente t2_dir_thr_120)  
- un conjunto compacto de indicadores no redundantes  
- factores dominados por momentum, complementados con volatilidad  

Este enfoque maximiza la relación señal/ruido y mejora las condiciones para el aprendizaje de modelos robustos y generalizables.

# **12. Selección final de features**

**1. Criterios de selección**

La selección de variables se realizó en base a los siguientes criterios:

- Magnitud de la señal fuera de muestra (IC OOS)
- Robustez (bajo gap entre IS y OOS)
- Consistencia de signo entre IS y OOS
- Presencia en múltiples regímenes de mercado
- Baja redundancia (controlada mediante análisis de correlación)

El objetivo es construir un conjunto de features compacto que capture la señal real del problema, evitando duplicación de información y mejorando la capacidad de generalización del modelo.

**2. Features principales (core)**

A partir del análisis conjunto de ranking, robustez, repetición entre regímenes y eliminación de redundancia, se seleccionan como núcleo del modelo los siguientes indicadores:

- roc_60  
- stoch_k_30  

Estos indicadores presentan:

- Mayor magnitud de IC OOS dentro del conjunto evaluado (especialmente roc_60)
- Presencia consistente en todos los regímenes de mercado
- Bajo gap entre IS y OOS
- Consistencia de signo en todos los análisis
- Representatividad dentro de sus respectivos grupos de redundancia

Desde el punto de vista estructural, capturan:

- Momentum de medio plazo (roc_60)
- Dinámicas de reversión / sobrecompra-sobreventa (stoch_k_30)

Este conjunto concentra la mayor parte de la señal direccional del sistema.

**3. Features complementarias**

Se identifican como variables complementarias:

- atr_norm_10  

Este indicador:

- Presenta IC positivo consistente (≈ 0.07 – 0.08 en horizontes relevantes)
- No es redundante con momentum u osciladores
- Aporta información distinta, asociada a la magnitud del movimiento (volatilidad)

Su rol es complementar la señal principal, no reemplazarla.

Opcionalmente, se pueden considerar:

- roc_30  

que:

- mantiene buena señal
- pero es parcialmente redundante con roc_60

Su inclusión debe validarse empíricamente en el modelo.

**4. Features descartadas**

Se descartan los siguientes grupos de indicadores:

- Volume ratios (volume_ratio_*)  
- MACD  
- MOM (mom_*)  
- RSI adicionales (rsi_*)  
- Bandas de Bollinger (bb_*)  
- EMAs (ema_*)  

Motivos:

- Baja o nula señal fuera de muestra (especialmente volumen y MACD)
- Alta redundancia dentro de sus grupos (BB, EMA, MOM)
- Información ya capturada por los indicadores seleccionados
- Correlaciones cercanas a 1 en múltiples casos

Esto confirma que muchos indicadores técnicos son transformaciones equivalentes de una misma dinámica subyacente.

**5. Feature set final**

Se proponen dos configuraciones:

Versión mínima (recomendada):

roc_60, stoch_k_30, atr_norm_10  

Versión extendida:

roc_60, stoch_k_30, atr_norm_10, roc_30  

La versión mínima prioriza simplicidad y robustez.  
La versión extendida incorpora mayor capacidad expresiva, a costa de cierta redundancia.

**6. Conclusión**

El análisis demuestra que el problema tiene una baja dimensionalidad efectiva. Aunque el dataset original contiene una gran cantidad de indicadores, la señal está concentrada en un número muy reducido de factores.

En particular:

- la señal principal está dominada por momentum (ROC)
- los osciladores aportan información complementaria
- la volatilidad agrega contexto relevante
- el resto de indicadores no aporta valor adicional significativo

En consecuencia, el modelo debe construirse sobre un conjunto pequeño de features no redundantes, priorizando calidad de señal y robustez sobre cantidad de variables.

# **13. Interpretación económica de los factores**

13. Interpretación económica de los factores

Introducción

En esta etapa se busca dar una interpretación económica a los indicadores seleccionados, conectando los resultados cuantitativos (IC) con el comportamiento esperado del mercado.

El análisis se realiza en dos niveles:

1) Clasificación teórica  
Cada indicador se asigna a una categoría según la dinámica que representa:

- Momentum → continuación del movimiento  
- Mean reversion → reversión del movimiento  
- Volatilidad → magnitud o probabilidad de movimiento  

2) Validación empírica  
Se contrasta esta clasificación con los datos mediante el Information Coefficient (IC), evaluando:

- el signo del IC  
- la coherencia con el comportamiento esperado  

Cómo interpretar los resultados

El IC mide la relación entre el valor actual del indicador y el movimiento futuro definido por el target.

La interpretación es la siguiente:

- IC positivo → valores altos del indicador se asocian con movimientos positivos futuros  
- IC negativo → valores altos del indicador se asocian con movimientos negativos futuros  

Para cada tipo de indicador:

- Momentum:
  - IC esperado: negativo en este contexto  
  - interpretación: movimientos recientes tienden a revertirse (mean-reversion dominante intradía)

- Mean reversion:
  - IC esperado: negativo  
  - interpretación: niveles altos (sobrecompra) anticipan caídas

- Volatilidad:
  - IC esperado: positivo  
  - interpretación: mayor volatilidad aumenta la probabilidad de movimientos significativos

La columna `matches_expectation` permite verificar si el comportamiento observado coincide con la teoría.


## **13.1. Código**

In [ ]:
import pandas as pd

feature_types = {
    "roc_60": "momentum",
    "roc_30": "momentum",
    "stoch_k_30": "mean_reversion",
    "atr_norm_10": "volatility",
}

expected_sign = {
    "momentum": "negative",
    "mean_reversion": "negative",
    "volatility": "positive",
}

interpretation_rows = []

for f in feature_types:
    for target in targets_cols:
        ic = compute_ic_by_day(
            mnq_intraday_t2_ti,
            f,
            target
        )

        if pd.isna(ic):
            sign = pd.NA
        elif ic > 0:
            sign = "positive"
        elif ic < 0:
            sign = "negative"
        else:
            sign = "zero"

        interpretation_rows.append({
            "feature": f,
            "type": feature_types[f],
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
            "sign": sign,
            "expected_sign": expected_sign[feature_types[f]],
            "matches_expectation": sign == expected_sign[feature_types[f]] if pd.notna(sign) else pd.NA,
        })

interpretation_df = pd.DataFrame(interpretation_rows)

interpretation_df = interpretation_df.sort_values(
    ["target", "abs_IC"],
    ascending=[True, False]
).reset_index(drop=True)

interpretation_df

,feature,type,target,IC,abs_IC,sign,expected_sign,matches_expectation
0,roc_60,momentum,t2_dir_thr_120,-0.160679,0.160679,negative,negative,True
1,roc_30,momentum,t2_dir_thr_120,-0.117993,0.117993,negative,negative,True
2,stoch_k_30,mean_reversion,t2_dir_thr_120,-0.092760,0.092760,negative,negative,True
3,atr_norm_10,volatility,t2_dir_thr_120,0.077469,0.077469,positive,positive,True
4,roc_60,momentum,t2_dir_thr_30,-0.077921,0.077921,negative,negative,True
5,atr_norm_10,volatility,t2_dir_thr_30,0.066986,0.066986,positive,positive,True
6,roc_30,momentum,t2_dir_thr_30,-0.052499,0.052499,negative,negative,True
7,stoch_k_30,mean_reversion,t2_dir_thr_30,-0.041289,0.041289,negative,negative,True
8,roc_60,momentum,t2_dir_thr_60,-0.102934,0.102934,negative,negative,True
9,roc_30,momentum,t2_dir_thr_60,-0.075575,0.075575,negative,negative,True


## **13.2. Conclusiones**



**1. Consistencia total entre teoría y datos**

Todos los indicadores presentan coincidencia entre el signo observado del IC y el signo esperado (`matches_expectation = True` en todos los casos). Esto valida tanto la selección de features como su interpretación económica.

---

**2. Dominancia de dinámica de reversión**

Los indicadores de momentum (roc_60, roc_30) presentan IC negativo en todos los horizontes, lo que indica que:

- el mercado intradía no está dominado por continuación de tendencia  
- sino por reversión a la media en horizontes de 30 a 120 minutos  

Esto es consistente con mercados líquidos donde los movimientos extremos tienden a corregirse.

---

**3. Confirmación del rol de osciladores**

El indicador stoch_k_30 también presenta IC negativo en todos los targets, lo que refuerza:

- la presencia de dinámicas de sobrecompra/sobreventa  
- la validez de señales de reversión basadas en osciladores  

Su comportamiento es coherente y estable en todos los horizontes.

---

**4.Rol complementario de la volatilidad**

El indicador atr_norm_10 presenta IC positivo en todos los targets, lo que indica que:

- mayores niveles de volatilidad están asociados a mayor probabilidad de alcanzar los umbrales del target T2  
- no aporta dirección, pero sí contexto sobre la intensidad del movimiento  

Esto lo convierte en una variable complementaria clave.

---

**5. Dependencia del horizonte**

Se observa que la magnitud del IC aumenta con el horizonte:

- valores más altos en t2_dir_thr_120  
- valores intermedios en t2_dir_thr_90  
- valores más bajos en horizontes cortos  

Esto confirma que:

- la señal se desarrolla con el tiempo  
- horizontes más largos permiten capturar mejor las dinámicas del mercado  

---

**Conclusión general**

El análisis confirma que los factores seleccionados no solo son estadísticamente relevantes, sino también económicamente coherentes.

En particular:

- el mercado presenta comportamiento de reversión a la media en los horizontes analizados  
- los indicadores de momentum actúan como señales contrarias (contrarian)  
- los osciladores refuerzan esta dinámica  
- la volatilidad aporta contexto sobre la probabilidad de eventos  

Esto valida que el modelo no está aprendiendo relaciones espurias, sino estructuras reales del mercado, lo que aumenta la probabilidad de generalización fuera de muestra.

# **14. Análisis de features de régimen**

## **14.1. Marco teórico**

Las variables de **régimen sí pueden aportar valor real**.

---

**1. Diferencia clave con `day_of_week`**

* `day_of_week` → constante dentro del día → inútil para IC intradía, eliminalos los flags de día.

* `regime` → cambia dentro del día → **sí tiene variabilidad**

Por eso **sí pueden influir en el target**
Los resultados ya mostraron que:

* el comportamiento cambia por régimen
* el IC cambia por régimen

Entonces: **el régimen explica parte de la dinámica del mercado**

---

**2. Decisión importante**

Decidimos entrenar un solo modelo porque sería más preciso y menos complejo. Además, es lo más simple y más robusto.

Elegimos un modelo único, por lo que el régimen debe entrar como feature

---

**3. Cómo incluirlo correctamente**

- No usaremos 5 flags (mal) por los siguientes problemas:

  * redundancia
  * colinealidad
  * ruido innecesario


Y lo recomendado para modelos es incluir una versión númerica:

```python
regime_map = {
    "is_overnight": 0,
    "is_premarket": 1,
    "is_opening": 2,
    "is_regular": 3,
    "is_closing": 4,
}

mnq_intraday_targets["regime_id"] = (
    mnq_intraday_targets[regime_cols]
    .idxmax(axis=1)
    .map(regime_map)
)
```

---

**4. ¿Aporta señal realmente?**

Aunque no midamos IC directamente, anteriormente ya lo validamos indirectamente:

  * IC cambia por régimen
  * ranking cambia por régimen
  * robustez cambia por régimen

- Eso prueba que el régimen contiene información

---

**5. Interpretación correcta**

El régimen le dice al modelo:

* contexto de liquidez
* volatilidad
* comportamiento estructural

Ejemplo:

* opening → más volatilidad
* overnight → más reversión
* regular → más estabilidad

---

**6. Decisión final**

* Mantenemos régimen como feature
* Convertirmos a una sola columna (`regime_id`)
* No usamos múltiples flags

---

**Conclusión simple**

* A diferencia de los días, **sí aporta señal**
* Es un feature **contextual importante**
* Debe estar en el modelo si no segmentas por régimen

# **15. Tratamiento de variables OHLCV**

**1. Punto de partida**

El dataset contiene las variables OHLCV en bruto:

- `open`, `high`, `low`, `close`, `volume`

Sin embargo, en el enfoque actual:

- los targets T2 se construyen a partir de movimientos futuros del precio (derivados de close)  
- los indicadores técnicos seleccionados (`roc_60`, `stoch_k_30`, `atr_norm_10`, etc.) también se derivan del precio y, en algunos casos, del volumen  

Por lo tanto, es necesario evaluar si las variables OHLCV en bruto aportan información adicional o si su contenido ya está representado en las features seleccionadas.

Este análisis se realiza en dos dimensiones:

- capacidad predictiva (IC respecto a los targets T2)  
- redundancia (correlación con indicadores técnicos seleccionados)  

---

**2. Problema con OHLC en bruto**

Las variables open, high, low y close en valores absolutos presentan limitaciones importantes:

- dependen del nivel de precio (no son invariantes a escala)  
- no son estacionarias  
- pueden inducir al modelo a aprender el nivel de precio en lugar de la dinámica  
- presentan alta correlación entre sí  
- son redundantes con indicadores derivados del precio  

En particular, la variable close requiere especial atención:

- los targets T2 se construyen directamente a partir de movimientos del precio  
- los principales indicadores seleccionados (ROC, Stochastic) también derivan de close  

Esto implica que incluir `close` en bruto introduce riesgo de:

- fuga de información estructural  
- aprendizaje de relaciones triviales  
- pérdida de capacidad de generalización  

3. Qué información ya está capturada

La información contenida en OHLC ya fue transformada en indicadores más robustos:

- Momentum: `roc_60`, `roc_30`  
- Reversión: `stoch_k_30`  
- Volatilidad: `atr_norm_10`  

Estas transformaciones:

- eliminan dependencia del nivel de precio  
- capturan relaciones dinámicas relevantes  
- son más estables estadísticamente  
- presentan mejor comportamiento fuera de muestra (validado por IC y robustez)  

Por lo tanto, el modelo ya dispone de representaciones superiores de la información contenida en OHLC.

---

**4. Evaluación empírica (IC y redundancia)**

Al evaluar las variables OHLC frente a los targets T2:

- su IC es significativamente menor que el de los indicadores técnicos seleccionados  
- su señal es inestable y altamente dependiente del nivel de precio  
- presentan alta correlación con indicadores derivados (especialmente ROC y EMA)  

En términos de redundancia:

- open, high, low y close están fuertemente correlacionados entre sí  
- también están indirectamente correlacionados con indicadores de tendencia y momentum  

Esto confirma que no aportan información nueva relevante.

---

**5. Caso particular de volume**

El volumen presenta un tratamiento distinto:

- en bruto, su señal es débil o nula (IC cercano a cero)  
- depende de escala y condiciones específicas del mercado  

En este trabajo:

- las transformaciones de volumen (volume_ratio_*) no mostraron señal significativa  
- el volumen no aporta valor predictivo relevante para los targets T2  

Por lo tanto:

- no se justifica incluir volume en bruto  
- tampoco se justifica incluir sus transformaciones en el modelo final  

---

**6. Criterio de decisión**

Se adopta el siguiente criterio:

- eliminar open, high, low, close  
- eliminar volume  
- conservar únicamente indicadores derivados con validación empírica (IC + robustez)  

Esto permite:

- reducir dimensionalidad  
- evitar multicolinealidad  
- eliminar dependencia del nivel de precio  
- mejorar la estabilidad del modelo  
- trabajar con variables económicamente interpretables  

---

**7. Conclusión**

Las variables OHLCV en bruto no se incorporan al modelo final, ya que:

- su información ya está contenida en los indicadores técnicos seleccionados  
- no aportan señal adicional relevante  
- introducen redundancia y posibles problemas de generalización  

En consecuencia, el modelo se construirá exclusivamente sobre features derivadas (momentum, reversión y volatilidad), que han demostrado:

- mayor capacidad predictiva  
- mayor robustez fuera de muestra  
- mejor interpretación económica  

Este enfoque permite maximizar la relación señal/ruido y mantener un conjunto de variables compacto y eficiente.

## **15.1. Código**

In [ ]:
rows = []

for feature in ohlcv_cols:
    for target in targets_cols:
        ic = compute_ic_by_day(
            mnq_intraday_t2_ti,
            feature,
            target
        )

        rows.append({
            "feature": feature,
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
        })

ohlcv_ic_df = (
    pd.DataFrame(rows)
    .sort_values(["target", "abs_IC"], ascending=[True, False])
    .reset_index(drop=True)
)

ohlcv_ic_df

,feature,target,IC,abs_IC
0,close,t2_dir_thr_120,-0.394029,0.394029
1,low,t2_dir_thr_120,-0.392744,0.392744
2,high,t2_dir_thr_120,-0.390991,0.390991
3,open,t2_dir_thr_120,-0.389540,0.389540
4,volume,t2_dir_thr_120,0.050545,0.050545
5,close,t2_dir_thr_30,-0.227981,0.227981
6,low,t2_dir_thr_30,-0.227434,0.227434
7,high,t2_dir_thr_30,-0.226125,0.226125
8,open,t2_dir_thr_30,-0.225556,0.225556
9,volume,t2_dir_thr_30,0.043417,0.043417


**Conclusiones del análisis de IC para variables OHLCV**

El análisis del Information Coefficient aplicado a las variables OHLCV muestra resultados que, si bien a primera vista parecen altamente favorables, requieren una interpretación cuidadosa desde el punto de vista metodológico.

---

**1. Magnitud elevada del IC en variables OHLC**

Las variables `close`, `open`, `high` y `low` presentan valores de IC significativamente elevados en todos los horizontes analizados (30, 60, 90 y 120), alcanzando magnitudes entre 0.22 y 0.39 en valor absoluto.

Esto las posiciona, en términos puramente numéricos, por encima de los indicadores técnicos previamente seleccionados.

Sin embargo, esta aparente superioridad no refleja necesariamente una mayor capacidad predictiva real.

---

**2. Presencia de leakage estructural**

El alto IC observado en las variables OHLC se explica principalmente por la forma en que se construyen los targets T2.

Dado que estos targets dependen directamente del precio futuro (derivado del `close`), incluir variables como `close` o cualquier otra derivada directa del precio introduce una relación estructural con el target.

Esto implica que el modelo puede aprender relaciones triviales basadas en el nivel actual del precio, en lugar de capturar patrones genuinos del mercado.

En consecuencia, el IC de estas variables se encuentra artificialmente inflado y no representa señal robusta fuera de muestra.

---

**3. Redundancia entre variables OHLC**

Las variables `open`, `high`, `low` y `close` presentan valores de IC prácticamente idénticos entre sí en todos los horizontes.

Esto indica que contienen esencialmente la misma información desde el punto de vista predictivo.

La inclusión simultánea de estas variables generaría:

* alta multicolinealidad
* redundancia informativa
* incremento innecesario de la dimensionalidad

sin aportar valor adicional al modelo.

---

**4. Comportamiento del volumen**

La variable `volume` presenta valores de IC bajos pero consistentes (en torno a 0.04–0.05).

Esto sugiere que su relación con el target es débil, aunque potencialmente más genuina que la de los precios.

Sin embargo, su capacidad predictiva es limitada y no constituye una fuente principal de señal.

---

**5. Interpretación global**

El análisis evidencia que las variables OHLC presentan un alto IC debido a su relación directa con la construcción del target, y no por capturar dinámicas estructurales del mercado.

Por el contrario, los indicadores técnicos previamente seleccionados ofrecen representaciones transformadas del precio que eliminan la dependencia del nivel absoluto y permiten capturar patrones más estables y generalizables.

---

**6. Conclusión**

Las variables OHLC en bruto no deben incorporarse al modelo, a pesar de su alto IC, debido a:

* la presencia de leakage estructural
* su alta redundancia
* la falta de valor predictivo real fuera de muestra

El volumen, aunque no presenta leakage, muestra una señal débil y no constituye un factor relevante en el contexto actual.

En consecuencia, el modelo debe construirse exclusivamente sobre indicadores derivados, que capturan de forma más adecuada la dinámica del mercado y presentan mayor robustez.


In [ ]:
selected_indicators = [
    "roc_60",
    "roc_30",
    "stoch_k_30",
    "atr_norm_10",
]

features_to_compare = ohlcv_cols + selected_indicators

rows = []

for feature in features_to_compare:
    for target in targets_cols:
        ic = compute_ic_by_day(
            mnq_intraday_t2_ti,
            feature,
            target
        )

        rows.append({
            "feature": feature,
            "feature_type": "OHLCV" if feature in ohlcv_cols else "indicator",
            "target": target,
            "IC": ic,
            "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
        })

compare_ohlcv_vs_indicators_df = (
    pd.DataFrame(rows)
    .sort_values(["target", "abs_IC"], ascending=[True, False])
    .reset_index(drop=True)
)

compare_ohlcv_vs_indicators_df

,feature,feature_type,target,IC,abs_IC
0,close,OHLCV,t2_dir_thr_120,-0.394029,0.394029
1,low,OHLCV,t2_dir_thr_120,-0.392744,0.392744
2,high,OHLCV,t2_dir_thr_120,-0.390991,0.390991
3,open,OHLCV,t2_dir_thr_120,-0.389540,0.389540
4,roc_60,indicator,t2_dir_thr_120,-0.160679,0.160679
5,roc_30,indicator,t2_dir_thr_120,-0.117993,0.117993
6,stoch_k_30,indicator,t2_dir_thr_120,-0.092760,0.092760
7,atr_norm_10,indicator,t2_dir_thr_120,0.077469,0.077469
8,volume,OHLCV,t2_dir_thr_120,0.050545,0.050545
9,close,OHLCV,t2_dir_thr_30,-0.227981,0.227981


**Comparación entre variables OHLCV e indicadores técnicos**

El análisis conjunto muestra que las variables OHLC (`open`, `high`, `low`, `close`) presentan los valores más altos de IC en todos los horizontes, superando ampliamente a los indicadores técnicos seleccionados.

Sin embargo, esta aparente superioridad no implica mayor capacidad predictiva real, ya que dichas variables están directamente relacionadas con la construcción del target T2. Esto introduce una dependencia estructural que inflaciona artificialmente el IC y no refleja una señal generalizable.

Por el contrario, los indicadores técnicos (`roc_60`, `roc_30`, `stoch_k_30`, `atr_norm_10`) presentan menor magnitud de IC, pero capturan transformaciones del precio que reflejan dinámica de mercado (momentum, reversión y volatilidad), lo que los hace más adecuados para el modelado.

Adicionalmente, se observa que el volumen mantiene una señal débil y consistente, sin aportar valor significativo.

En conclusión, aunque las variables OHLC dominan en términos de IC bruto, no deben ser utilizadas como features debido al riesgo de leakage y redundancia. Los indicadores técnicos seleccionados siguen siendo la base más robusta y metodológicamente correcta para el modelo.


## **15.3. Correlación OHLCV e indicadores técnicos**

In [ ]:
corr_matrix = (
    mnq_intraday_t2_ti[features_to_compare]
    .corr()
)

corr_matrix

,open,high,low,close,volume,roc_60,roc_30,stoch_k_30,atr_norm_10
open,1.000000,0.999999,0.999999,0.999998,0.177342,-0.003262,-0.002717,0.015877,-0.243940
high,0.999999,1.000000,0.999997,0.999999,0.178136,-0.003203,-0.002584,0.016069,-0.243086
low,0.999999,0.999997,1.000000,0.999999,0.176488,-0.003056,-0.002464,0.016287,-0.244797
close,0.999998,0.999999,0.999999,1.000000,0.177302,-0.002994,-0.002344,0.016473,-0.243939
volume,0.177342,0.178136,0.176488,0.177302,1.000000,-0.080175,-0.071468,-0.076464,0.549191
roc_60,-0.003262,-0.003203,-0.003056,-0.002994,-0.080175,1.000000,0.713680,0.472036,-0.065685
roc_30,-0.002717,-0.002584,-0.002464,-0.002344,-0.071468,0.713680,1.000000,0.665771,-0.045756
stoch_k_30,0.015877,0.016069,0.016287,0.016473,-0.076464,0.472036,0.665771,1.000000,-0.077149
atr_norm_10,-0.243940,-0.243086,-0.244797,-0.243939,0.549191,-0.065685,-0.045756,-0.077149,1.000000


## **15.2. Conclusión sobre variables OHLCV**

El análisis de correlación confirma de manera clara la estructura de redundancia y complementariedad entre las variables.

**1. Redundancia extrema en OHLC**

Las variables `open`, `high`, `low` y `close` presentan correlaciones prácticamente perfectas (~1.0) entre sí.

Esto indica que:

* contienen la misma información
* son completamente redundantes
* no tiene sentido incluir más de una

En la práctica, incluirlas todas generaría multicolinealidad severa sin aportar valor adicional.

---

**2. Independencia entre OHLC e indicadores**

Las correlaciones entre OHLC y los indicadores técnicos (`roc_*`, `stoch_k_30`) son cercanas a cero.

Esto implica que:

* los indicadores no dependen del nivel absoluto del precio
* capturan dinámica (movimiento), no escala

Este es un punto clave a favor de los indicadores técnicos.

---

**3. Redundancia parcial dentro de indicadores de momentum**

* `roc_60` y `roc_30` → correlación alta (~0.71)
* `roc_30` y `stoch_k_30` → correlación moderada-alta (~0.66)
* `roc_60` y `stoch_k_30` → correlación moderada (~0.47)

Esto indica que:

* comparten parte de la información (momentum / reversión)
* pero no son completamente redundantes

---

**4. Rol diferenciado de la volatilidad**

`atr_norm_10` muestra:

* baja correlación con indicadores de momentum
* correlación negativa moderada con OHLC (~ -0.24)
* correlación moderada con volumen (~0.55)

Esto confirma que:

* aporta información distinta
* captura dimensión de volatilidad

---

**5. Volumen como variable auxiliar**

El volumen presenta:

* baja correlación con indicadores
* correlación moderada con volatilidad

Esto sugiere que:

* está más relacionado con actividad de mercado que con dirección

---

**6. Conclusión**

El análisis de correlación refuerza que:

* OHLC es completamente redundante internamente
* los indicadores técnicos capturan información distinta y más útil
* existe cierta redundancia dentro del grupo de momentum, pero no total
* la volatilidad (`atr_norm_10`) aporta una dimensión complementaria

En consecuencia, el modelo debe:

* excluir OHLC en bruto
* mantener un conjunto reducido de indicadores
* evitar duplicación dentro de familias altamente correlacionadas
* combinar momentum + reversión + volatilidad para capturar distintas dinámicas del mercado.


# **16. Análisis de la variable `minute_of_day` y `regime_id`**

La variable `minute_of_day` representa la posición temporal de cada observación dentro de la jornada intradía, permitiendo capturar la estructura temporal del mercado a lo largo del día.

A diferencia de los indicadores técnicos, que buscan modelar directamente la relación entre variables de entrada y el movimiento futuro del precio, esta variable tiene un carácter contextual. Su objetivo no es predecir dirección por sí sola, sino aportar información sobre el entorno en el que se generan los movimientos.

En el contexto de los targets T2, esto es particularmente relevante, ya que la probabilidad de que el precio supere un determinado umbral no es uniforme a lo largo del día, sino que depende del régimen intradía y de la dinámica temporal del mercado.

En particular, `minute_of_day` permite capturar patrones conocidos como:

* mayor probabilidad de movimientos amplios durante la apertura
* cambios en la estructura de volatilidad a lo largo de la sesión
* diferencias entre períodos de baja y alta actividad

Por esta razón, su utilidad no necesariamente se refleja en un alto IC directo, sino en su capacidad de aportar contexto al modelo.

El análisis de esta variable se abordará en dos niveles:

* evaluación de su relación directa con el target mediante IC
* análisis del comportamiento del target segmentado por intervalos intradía

El objetivo es determinar si `minute_of_day` aporta valor como feature contextual, complementando a los indicadores técnicos, aun cuando su señal predictiva directa sea limitada.

## **16.1. Código**

In [ ]:
#MEDIR IC DIRECTO CON EL TARGET
rows = []

for target in targets_cols:
    ic = compute_ic_by_day(
        mnq_intraday_t2_ti,
        "minute_of_day",
        target
    )

    rows.append({
        "feature": "minute_of_day",
        "target": target,
        "IC": ic,
        "abs_IC": abs(ic) if pd.notna(ic) else pd.NA,
    })

minute_ic_df = pd.DataFrame(rows)
minute_ic_df

,feature,target,IC,abs_IC
0,minute_of_day,t2_dir_thr_30,-0.002889,0.002889
1,minute_of_day,t2_dir_thr_60,-0.001183,0.001183
2,minute_of_day,t2_dir_thr_90,-0.000305,0.000305
3,minute_of_day,t2_dir_thr_120,-0.004306,0.004306


Primeras observaciones:

* El IC es **prácticamente cero en todos los horizontes**
  → no hay relación monotónica directa con el target

* La señal es **mucho menor que cualquier indicador técnico**
  → no aporta capacidad predictiva directa

* El signo es levemente negativo pero **irrelevante**
  → no tiene interpretación práctica

* Esto confirma que `minute_of_day` **no predice por sí sola**
  → es una variable **contextual, no predictiva**

* Su valor potencial está en:
  → ayudar al modelo a diferenciar momentos del día
  → capturar efectos intradía combinados con otras features

**Conclusión:**
No sirve como feature independiente, pero puede aportar valor como variable de contexto dentro del modelo.


Luego, el paso 2 es analizar el comportamiento del target por tramos intradía. Para eso, primero creamos buckets:

In [ ]:
df = mnq_intraday_t2_ti.copy()

df["minute_bucket"] = pd.cut(
    df["minute_of_day"],
    bins=6,
    labels=[f"bucket_{i}" for i in range(6)]
)

#df[["minute_of_day", "minute_bucket"]].head()

,minute_of_day,minute_bucket
datetime,,
2020-01-02 05:59:00-05:00,359,bucket_0
2020-01-02 06:00:00-05:00,360,bucket_0
2020-01-02 06:01:00-05:00,361,bucket_0
2020-01-02 06:02:00-05:00,362,bucket_0
2020-01-02 06:03:00-05:00,363,bucket_0


In [ ]:
results = []

for target in targets_cols:
    tmp = df[[target, "minute_bucket"]].dropna().copy()

    summary = (
        tmp.groupby("minute_bucket")[target]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .rename(columns={
            -1: "pct_down",
             0: "pct_neutral",
             1: "pct_up"
        })
        .reset_index()
    )

    summary["target"] = target

    results.append(summary)

minute_bucket_analysis = pd.concat(results, ignore_index=True)

minute_bucket_analysis


,minute_bucket,pct_down,pct_neutral,pct_up,target
0,bucket_0,0.058083,0.890875,0.051042,t2_dir_thr_30
1,bucket_1,0.123421,0.751197,0.125382,t2_dir_thr_30
2,bucket_2,0.244502,0.494139,0.261359,t2_dir_thr_30
3,bucket_3,0.162834,0.672309,0.164857,t2_dir_thr_30
4,bucket_4,0.146154,0.706826,0.147019,t2_dir_thr_30
5,bucket_5,0.152624,0.697428,0.149948,t2_dir_thr_30
6,bucket_0,0.064139,0.885454,0.050407,t2_dir_thr_60
7,bucket_1,0.160602,0.667714,0.171683,t2_dir_thr_60
8,bucket_2,0.221853,0.536023,0.242124,t2_dir_thr_60
9,bucket_3,0.150819,0.700463,0.148718,t2_dir_thr_60


### **Comparación de buckets contra feature `regime_id`**

In [ ]:
df = mnq_intraday_t2_ti.copy()

targets = [
    "t2_dir_thr_30",
    "t2_dir_thr_60",
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

results = []

for target in targets:

    # -------- 1. Por regime_id --------
    reg = (
        df.groupby("regime_id")[target]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .rename(columns={
            -1: "pct_down",
             0: "pct_neutral",
             1: "pct_up"
        })
        .reset_index()
    )
    reg["type"] = "regime"
    reg["target"] = target

    # -------- 2. Por minute_bucket --------
    tmp = df.copy()

    tmp["minute_bucket"] = pd.cut(
        tmp["minute_of_day"],
        bins=6,
        labels=[f"bucket_{i}" for i in range(6)]
    )

    buck = (
        tmp.groupby("minute_bucket")[target]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .rename(columns={
            -1: "pct_down",
             0: "pct_neutral",
             1: "pct_up"
        })
        .reset_index()
        .rename(columns={"minute_bucket": "regime_id"})
    )
    buck["type"] = "bucket"
    buck["target"] = target

    results.append(pd.concat([reg, buck], ignore_index=True))

comparison_df = pd.concat(results, ignore_index=True)

comparison_df

,regime_id,pct_down,pct_neutral,pct_up,type,target
0,0,0.071181,0.860666,0.068153,regime,t2_dir_thr_30
1,1,0.173372,0.652201,0.174427,regime,t2_dir_thr_30
2,2,0.253797,0.469073,0.277130,regime,t2_dir_thr_30
3,3,0.159393,0.680366,0.160242,regime,t2_dir_thr_30
4,4,0.197683,0.596139,0.206178,regime,t2_dir_thr_30
5,bucket_0,0.058083,0.890875,0.051042,bucket,t2_dir_thr_30
6,bucket_1,0.123421,0.751197,0.125382,bucket,t2_dir_thr_30
7,bucket_2,0.244502,0.494139,0.261359,bucket,t2_dir_thr_30
8,bucket_3,0.162834,0.672309,0.164857,bucket,t2_dir_thr_30
9,bucket_4,0.146154,0.706826,0.147019,bucket,t2_dir_thr_30


## **16.2. Conclusiones del análisis de `minute_of_day` vs `regime_id`**




El análisis comparativo entre la segmentación intradía mediante `minute_bucket` y la variable `regime_id` muestra que ambas capturan prácticamente la misma estructura del comportamiento del target T2 a lo largo del día.

En todos los horizontes analizados, las distribuciones de probabilidad de las clases (-1, 0, 1) son muy similares entre los buckets y los regímenes correspondientes. En particular, se observa un patrón consistente:

* alta proporción de neutralidad al inicio del día
* incremento significativo de movimientos en la zona media
* estabilización hacia el final de la sesión

Este patrón se reproduce de forma casi idéntica tanto en `minute_bucket` como en `regime_id`, lo que indica que la información temporal relevante ya está correctamente capturada por la variable de régimen.

Si bien `minute_bucket` ofrece una discretización ligeramente más granular del tiempo, esta mayor resolución no introduce cambios sustanciales en la distribución del target, por lo que no aporta información adicional significativa.

En consecuencia, se concluye que `regime_id` es suficiente para modelar el efecto intradía, y que la inclusión de `minute_of_day` o de variables derivadas como `minute_bucket` resultaría redundante.

Por lo tanto, para el modelo final se decide:

* mantener `regime_id` como variable temporal
* excluir `minute_of_day` y su discretización

Esta decisión permite simplificar el modelo sin pérdida de información relevante, evitando además redundancia y posible sobreajuste.


# **17. Cierre del stage de selección de features**

**1. Features seleccionadas**

A partir del análisis de señal, robustez, consistencia y redundancia, se define el siguiente conjunto final de variables:

Features estructurales (señal directa):

* roc_60
* roc_30
* stoch_k_30
* atr_norm_10

Features contextuales:

* regime_id

---

**2. Features eliminadas**

Durante el proceso se descartaron las siguientes variables:

* Variables OHLC en bruto (open, high, low, close)
* Volumen en bruto (volume)
* Variables de tiempo (minute_of_day)

Las razones principales fueron:

* presencia de relaciones triviales con el target (leakage estructural en OHLC)
* alta redundancia entre variables
* falta de estacionariedad (precios en niveles absolutos)
* ausencia de señal predictiva directa (caso de minute_of_day)
* disponibilidad de representaciones más informativas mediante indicadores técnicos

---

**3. Estructura del feature set final**

El conjunto final combina:

* indicadores técnicos que capturan la dinámica del precio:

  * momentum (roc_60, roc_30)
  * reversión (stoch_k_30)
  * volatilidad (atr_norm_10)

* una variable contextual:

  * regime_id, que captura la estructura intradía del mercado

Esta combinación permite al modelo:

* identificar patrones de movimiento relevantes
* adaptarse a distintos regímenes de mercado
* evitar dependencia de variables redundantes o no generalizables

---

**4. Targets seleccionados**

Se seleccionan como targets finales:

* t2_dir_thr_90
* t2_dir_thr_120

La elección se basa en los siguientes criterios:

* mayor magnitud de IC en los indicadores técnicos
* mejor estabilidad entre IS y OOS
* señal más consistente entre regímenes de mercado
* menor impacto del ruido intradía presente en horizontes cortos

Estos horizontes representan movimientos más estructurales del mercado, donde la señal es más clara y robusta.

---

**5. Validación del proceso**

El feature set final cumple con los criterios definidos:

* presenta señal out-of-sample consistente
* muestra robustez entre IS y OOS (gap reducido)
* mantiene consistencia de signo en todos los análisis
* evita redundancia y multicolinealidad
* está alineado con la interpretación económica de los factores

---

**6. Conclusión**

Se obtiene un conjunto de variables compacto, informativo y coherente con la dinámica del mercado intradía.

La señal se concentra en pocos factores robustos, principalmente asociados a momentum, reversión y volatilidad, complementados por el contexto intradía a través de `regime_id`.

El dataset resultante queda preparado para las siguientes etapas del pipeline:

* partición temporal de datos
* construcción de ventanas
* entrenamiento y evaluación de modelos predictivos

Este enfoque prioriza calidad sobre cantidad de features, favoreciendo la generalización y reduciendo el riesgo de sobreajuste.
